# Agentic Multi-Agent Crisis CDS — LangGraph

Crisis clinical decision support under incomplete information. This notebook holds the
full pipeline as importable definitions; variants load it via `%run` with
`CDS_IMPORT_ONLY=1` (which skips the demo cells at the bottom).

**Run All** — every setup cell (§1–§12) is cheap and runs automatically. The **End-to-End Demo** cell at the end is **off by default**; set `ENABLE_DEMO = True` to run one full case (~150 s, real tokens), then re-run just that cell.

**Sections:** 1 Config · 2 AgentHandoff · 3 State · 4 Intake · 5 FAISS & RAG · 6 Tools ·
7 Parallel safety · 8 Orchestrator · 9 Critique · 10 Decision memory · 11 Graph · 12 Run summary

## 1 · Imports, Configuration & Secrets

Secrets from project `.env` (no hardcoded keys); single `MODEL_ID`; pooled `_HTTP`
session reused for all RxNav / OpenFDA calls; absolute data paths.


In [ ]:
# Dependencies (uncomment to install)
# !pip install langchain-openai langchain langgraph faiss-cpu sentence-transformers python-dotenv pandas numpy

import os, json, re, uuid, time, itertools, gzip
from typing import Annotated, Any, Optional, Literal, TypedDict
from dataclasses import dataclass, field
from operator import add
from datetime import datetime
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import faiss
import pandas as pd
import requests
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    BaseMessage, HumanMessage, SystemMessage, AIMessage, ToolMessage
)
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

# Secrets
# Credentials are read from the project .env — never hardcode keys in the notebook.
# .env keys: OPENAI_API_KEY, HUGGINGFACE_API_KEY (also LANGCHAIN_API_KEY, GROQ_API_KEY).
load_dotenv("D:/Projects/Projects/agents/.env", override=True)
load_dotenv(override=False)  # fallback: any .env in CWD / parents
# Cell §9's HF critique client expects HF_TOKEN; map it from HUGGINGFACE_API_KEY.
os.environ.setdefault("HF_TOKEN", os.getenv("HUGGINGFACE_API_KEY", ""))

assert os.getenv("OPENAI_API_KEY"), \
    "OPENAI_API_KEY missing — add it to D:/Projects/Projects/agents/.env"

# Model
# One backbone for the whole pipeline (paper: "GPT-5-nano"); keep MODEL_ID and
# the manuscript in sync.
MODEL_ID = "gpt-5-nano"
# gpt-5-nano is a REASONING model: reasoning tokens dominate latency.
# Two tiers. The orchestrator router MUST stay medium — both "minimal"
# and "low" make it loop one tool until the iteration cap (verified
# 2026-05-18). Only leaf JSON tasks tolerate minimal.
#   LLM       minimal — leaf tools, RAG grading/rewrite, critique
#   LLM_THINK medium  — parse_intake, orchestrator router, triage
LLM = ChatOpenAI(model=MODEL_ID, temperature=0, max_tokens=8192,
                 timeout=120, max_retries=3,
                 reasoning_effort="minimal")  # leaf tools / RAG / critique
LLM_THINK = ChatOpenAI(model=MODEL_ID, temperature=0, max_tokens=8192,
                       timeout=120, max_retries=3,
                       reasoning_effort="medium")  # intake+router+triage

# HTTP session
# Connection pooling + keep-alive for the repeated RxNav / OpenFDA calls in §6.
_HTTP = requests.Session()
_HTTP.headers.update({"User-Agent": "TriAgent-CDS/1.0"})

# Paths
DATA_DIR  = Path("D:/Projects/Projects/agents/data")
SYN_DIR   = DATA_DIR / "synthetic data"
REAL_DIR  = DATA_DIR / "real data"
CACHE_DIR = DATA_DIR / "cache"

print(f"Model      : {MODEL_ID}")
print(f"OpenAI key : {'set' if os.getenv('OPENAI_API_KEY') else 'MISSING'}")
print(f"HF token   : {'set' if os.getenv('HF_TOKEN') else 'MISSING'}")
print(f"Data root  : {DATA_DIR}")

## 2 · AgentHandoff Protocol

Uniform 8-field envelope (incl. `input_received` and `timestamp`) returned by every
tool and appended verbatim to the audit trace.


In [ ]:
@dataclass
class AgentHandoff:
    """Structured envelope returned by every tool. Guarantees 100% audit completeness."""
    tool_name:           str
    input_received:      dict
    reasoning_trace:     str
    output:              dict
    confidence:          float        # 0.0 – 1.0
    caveats:             list
    suggested_next_tools: list
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())

    def to_dict(self) -> dict:
        return {
            "tool_name":            self.tool_name,
            "input_received":       self.input_received,
            "reasoning_trace":      self.reasoning_trace,
            "output":               self.output,
            "confidence":           self.confidence,
            "caveats":              self.caveats,
            "suggested_next_tools": self.suggested_next_tools,
            "timestamp":            self.timestamp,
        }

print("AgentHandoff protocol defined.")

## 3 · State Schema

`CrisisOrchestratorState` (`TypedDict`) with reducers — `merge_dict` (case state),
`add` (audit trace / flags), `add_messages` (ReAct history) — plus per-tool result
slots and control flags (`is_emergency`, `critique_retry_count`, `skip_critique`).


In [ ]:
def merge_dict(a: dict, b: dict) -> dict:
    return {**a, **b}

class CrisisOrchestratorState(TypedDict, total=False):
    # Core patient data
    case_state:          Annotated[dict, merge_dict]
    missing_data_flags:  Annotated[list, add]

    # ReAct loop conversation history
    messages:            Annotated[list[BaseMessage], add_messages]

    # Audit trail — append-only; every AgentHandoff.to_dict() goes here
    audit_trace:         Annotated[list[dict], add]

    # Tool outputs (populated as the ReAct loop progresses)
    triage_result:       Optional[dict]
    rag_result:          Optional[dict]
    treatment_plan:      Optional[dict]
    d3_result:           Optional[dict]
    allergy_result:      Optional[dict]
    safety_verdict:      Optional[dict]
    clinical_note:       Optional[dict]
    critique_report:     Optional[dict]
    emergency_payload:   Optional[dict]

    # Control
    thread_id:            str
    is_emergency:         bool
    critique_retry_count: int
    critique_feedback:    Optional[str]
    skip_critique:        bool

print("CrisisOrchestratorState defined.")

## 4 · Intake Parser (LLM-based, outside the ReAct loop)

Normalises any input into a canonical `case_state` before the Orchestrator runs.

- **Dict input** → fields passed through, no LLM call.
- **Free text** → one `LLM` strict-JSON call extracting symptoms (findings only),
  vitals (numeric; °F→°C; absent → `null`), medications, allergies (`reaction="unknown"`
  if unstated), and past medical history.
- Missing fields are **flagged** (`missing_vitals`, `missing_hr`, …), not rejected;
  on parse failure the raw text becomes a single symptom.


In [ ]:
def parse_intake(raw: Any) -> tuple[dict, list[str]]:
    """
    Accept structured dict or free-text string.
    Dicts use direct field extraction; free text uses LLM extraction (no regex).
    Missing fields are flagged rather than rejected.
    Returns (case_state, missing_data_flags).
    """
    missing: list[str] = []

    if isinstance(raw, dict):
        cs = {
            "symptoms":             raw.get("symptoms", []),
            "vitals":               raw.get("vitals", {}),
            "medications":          raw.get("medications", []),
            "allergies":            raw.get("allergies", []),
            "past_medical_history": raw.get("past_medical_history", []),
            "acuity_signals":       raw.get("acuity_signals", []),
        }
    else:
        system = """Extract structured clinical information from a patient narrative.
Return JSON only — no markdown, no explanation:
{
  "symptoms": ["<clinical finding>"],
  "vitals": {
    "bp_systolic":  <integer or null>,
    "bp_diastolic": <integer or null>,
    "hr":           <integer or null>,
    "rr":           <integer or null>,
    "temp_celsius": <float or null>,
    "spo2":         <integer or null>
  },
  "medications": [{"name": "<drug name>", "indication": "<reason or unknown>"}],
  "allergies":   [{"allergen": "<substance>", "reaction": "<reaction or unknown>"}],
  "past_medical_history": ["<diagnosed condition>"],
  "acuity_signals": ["<qualitative severity cue in the patient's own words>"]
}
Rules:
- symptoms: clinical findings only — chest pain, dyspnoea, fever. NOT location phrases like "I am in the ER".
- vitals: numbers only; convert Fahrenheit to Celsius (C=(F-32)*5/9, round to 1dp); null if absent.
- acuity_signals: qualitative red-flag cues to preserve acuity when numbers are absent —
  e.g. "cannot speak in full sentences", "gasping", "lips blue", "BP feels very low",
  "heart racing", "near-syncope / passing out", "confused / not making sense",
  "hard to stay awake", "cold and clammy", "chest pain at rest". Empty list if none.
- medications: real drug/supplement names only — NOT devices, procedures, or anatomical descriptions.
- allergies: substance + reaction if stated; use reaction="unknown" if reaction not mentioned.
- past_medical_history: confirmed diagnoses only, not current complaints."""

        def _try_parse(sys_prompt):
            resp     = LLM_THINK.invoke([
                SystemMessage(content=sys_prompt),
                HumanMessage(content=f"Patient narrative:\n{str(raw)}")
            ])
            raw_json = re.sub(r"```(?:json)?|```", "", resp.content).strip()
            return json.loads(raw_json)

        try:
            parsed = _try_parse(system)
            # gpt-5-nano sometimes returns valid JSON with empty symptoms when
            # reasoning consumes the output budget. A stricter retry recovers it
            # and prevents the cascade of low-confidence downstream tools.
            if not parsed.get("symptoms"):
                parsed = _try_parse(
                    system + "\n\nIMPORTANT: extract at least one symptom from "
                    "the narrative; clinical findings (e.g. confusion, dyspnoea, "
                    "pain, bleeding, weakness) MUST be listed as symptoms. "
                    "Return ONLY the JSON object.")
            vitals   = {k: v for k, v in parsed.get("vitals", {}).items() if v is not None}
            cs = {
                "symptoms":             parsed.get("symptoms", []),
                "vitals":               vitals,
                "medications":          parsed.get("medications", []),
                "allergies":            parsed.get("allergies", []),
                "past_medical_history": parsed.get("past_medical_history", []),
                "acuity_signals":       parsed.get("acuity_signals", []),
            }
        except Exception as e:
            print(f"  [intake] LLM parse failed ({e}) — using raw text as single symptom.")
            cs = {
                "symptoms":             [str(raw)],
                "vitals":               {},
                "medications":          [],
                "allergies":            [],
                "past_medical_history": [],
                "acuity_signals":       [],
            }

    # Lay narratives never carry numeric vitals, so per-vital sub-flags
    # (missing_bp_systolic/missing_hr/missing_spo2) fired on 100% of cases
    # and added no signal. Keep only the summary `missing_vitals` flag —
    # which the triage rubric already handles via its missing-data rule.
    for f in ("symptoms", "vitals", "medications", "allergies"):
        if not cs.get(f):
            missing.append(f"missing_{f}")

    return cs, missing

print("Intake parser defined (LLM-based for free text).")

## 5 · FAISS Index & Agentic RAG Subagent

**Corpus** — 49,000 real MIMIC-IV discharge notes (`rag_corpus_49k.jsonl`), built **patient-disjoint** from the evaluation set (`eval_1000_real.jsonl`), so the retrieval index and the evaluation set have **zero subject overlap** (verified at build time).

**Cache** — a binary `rag_index_v5_real_esi.faiss` (`IndexFlatIP`) plus `rag_meta_v5_real_esi.json`. First build embeds with progress + ETA; later runs load from cache.

**Loop** — `agentic_rag_retrieve(query, k=5, max_rewrites=1, threshold=0.5)`: retrieve → LLM relevance grade → query rewrite (up to 1 rewrite); sets `low_confidence` when the score stays below `threshold`.


### 5.1 · Corpus index — build & cache

In [ ]:
import textwrap
# Embeddings
try:
    from sentence_transformers import SentenceTransformer
    _EMBED_MODEL = SentenceTransformer("all-MiniLM-L6-v2")
    EMBED_DIM = 384
    def embed(text: str) -> np.ndarray:
        return _EMBED_MODEL.encode([text[:512]], normalize_embeddings=True).astype("float32")[0]
    print("Embeddings: sentence-transformers / all-MiniLM-L6-v2")
except ImportError:
    EMBED_DIM = 384
    def embed(text: str) -> np.ndarray:
        rng = np.random.default_rng(abs(hash(text)) % 2**31)
        v   = rng.standard_normal(EMBED_DIM).astype("float32")
        return v / (np.linalg.norm(v) + 1e-9)
    print("Embeddings: hash-based fallback (install sentence-transformers for semantic search)")

# Report index
# Corpus = data/rag_corpus_49k.jsonl, built by notebooks/build_real_dataset.py:
# 49,000 real MIMIC-IV discharge notes, PATIENT-DISJOINT from the 1,000 eval
# cases (0 subject overlap, verified at build time). We still assert note_id
# disjointness here as a second leakage guard before embedding.
RAG_CORPUS_JSONL = DATA_DIR / "rag_corpus_49k.jsonl"
EVAL_FILE        = DATA_DIR / "eval_1000_real.csv"

REPORT_INDEX: list[dict] = []
FAISS_INDEX:  Optional[faiss.IndexFlatIP] = None

def build_report_index(n_real: Optional[int] = None) -> None:
    """
    Build or load the FAISS RAG index from the real patient-disjoint corpus.
    Cache (v5 = real-ESI corpus; old v2/v4 caches are stale, do not reuse):
      rag_meta_v5_real_esi.json   - id/source/text metadata only
      rag_index_v5_real_esi.faiss - binary FAISS vectors (49k x 384-dim)
    `n_real` truncates the corpus (None = all 49k).
    """
    global REPORT_INDEX, FAISS_INDEX
    cache_meta  = CACHE_DIR / "rag_meta_v5_real_esi.json"
    cache_faiss = CACHE_DIR / "rag_index_v5_real_esi.faiss"

    if cache_meta.exists() and cache_faiss.exists():
        REPORT_INDEX = json.loads(cache_meta.read_text(encoding="utf-8"))
        FAISS_INDEX  = faiss.read_index(str(cache_faiss))
        print(f"Loaded cached index: {len(REPORT_INDEX):,} records | "
              f"{FAISS_INDEX.ntotal:,} vectors.")
        return

    if not RAG_CORPUS_JSONL.exists():
        raise FileNotFoundError(
            f"RAG corpus missing: {RAG_CORPUS_JSONL}. Run "
            "notebooks/build_real_dataset.py first."
        )
    if not EVAL_FILE.exists():
        raise FileNotFoundError(
            f"Eval file missing: {EVAL_FILE}. Cannot verify retrieval/eval "
            "separation - refusing to build a possibly leaky index."
        )

    eval_ids = set(
        pd.read_csv(EVAL_FILE, encoding="utf-8-sig",
                    usecols=["note_id"])["note_id"].astype(str)
    )
    print(f"  Separation: holding out {len(eval_ids):,} evaluation note_ids")

    records: list[dict] = []
    with RAG_CORPUS_JSONL.open(encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            d = json.loads(line)
            records.append({
                "id":     f"mimic/{d['note_id']}",
                "source": str(d["note_id"]),
                "text":   str(d["text"]),
            })
    if n_real is not None:
        records = records[:n_real]

    leaked = {r["source"] for r in records} & eval_ids
    assert not leaked, f"LEAKAGE: {len(leaked)} eval notes in retrieval index"
    print(f"  MIMIC-IV records: {len(records):,}  |  eval overlap: 0 (verified)")

    # Build embeddings
    total = len(records)
    print(f"  Embedding {total:,} records...")
    vecs: list[np.ndarray] = []
    t_embed = time.time()
    for i, rec in enumerate(records):
        vecs.append(embed(rec["text"][:800]))
        if (i + 1) % 5_000 == 0:
            elapsed_e = time.time() - t_embed
            eta       = elapsed_e / (i + 1) * (total - i - 1)
            print(f"    {i+1:>6,}/{total:,}  ({(i+1)/total*100:.0f}%)  ETA {eta/60:.1f}m")

    FAISS_INDEX = faiss.IndexFlatIP(EMBED_DIM)
    if vecs:
        FAISS_INDEX.add(np.vstack(vecs).astype("float32"))
    REPORT_INDEX = records

    # Save: binary FAISS + lightweight metadata JSON (no embeddings)
    CACHE_DIR.mkdir(exist_ok=True)
    faiss.write_index(FAISS_INDEX, str(cache_faiss))
    cache_meta.write_text(json.dumps(records), encoding="utf-8")
    size_mb = cache_faiss.stat().st_size / 1e6
    print(f"  Saved: {cache_faiss.name} ({size_mb:.1f} MB)  +  {cache_meta.name}")
    print(f"  Index ready: {len(records):,} records | {FAISS_INDEX.ntotal:,} vectors.")

### 5.2 · Retrieve · grade · rewrite

In [ ]:
def _faiss_search(query: str, k: int = 5) -> list[dict]:
    if not FAISS_INDEX or FAISS_INDEX.ntotal == 0:
        return []
    q = embed(query).reshape(1, -1)
    scores, idxs = FAISS_INDEX.search(q, min(k, FAISS_INDEX.ntotal))
    return [
        {"id": REPORT_INDEX[i]["id"], "source": REPORT_INDEX[i]["source"],
         "text_excerpt": REPORT_INDEX[i]["text"], "similarity": float(s)}
        for s, i in zip(scores[0], idxs[0]) if i >= 0
    ]


def _grade_one(query: str, idx: int, note: str) -> tuple[int, float, str]:
    """Grade a SINGLE retrieved note (in full) against the query."""
    if not note or not note.strip():
        return idx, 0.0, "empty note"
    prompt = (
        f"Rate the clinical relevance of this ONE retrieved discharge note to the "
        f"query (0.0-1.0).\n"
        f"Consider shared symptoms/diagnoses, treatment relevance, outcome "
        f"informativeness.\n\n"
        f"Query: {query}\n\nRetrieved note:\n{note}\n\n"
        f'Return JSON only: {{"score": <float 0-1>, "reasoning": "<2-3 sentences>"}}'
    )
    try:
        raw = LLM.invoke([HumanMessage(content=prompt)]).content
        raw = re.sub(r"```(?:json)?|```", "", raw).strip()
        p   = json.loads(raw)
        if isinstance(p, list) and p:
            p = p[0]
        return idx, float(p["score"]), str(p.get("reasoning", ""))
    except Exception as e:
        return idx, 0.5, f"grading failed: {e}"


def _grade_relevance(query: str, cases: list[dict]) -> tuple[float, str]:
    """Grade each retrieved note in its own LLM call, in parallel. The set
    score is the mean of the per-note scores; the reasoning leads with the
    weakest note so the query rewriter targets the actual gap."""
    if not cases:
        return 0.0, "No cases retrieved."
    with ThreadPoolExecutor(max_workers=min(len(cases), 5)) as exe:
        graded = sorted(exe.map(
            lambda ic: _grade_one(query, ic[0], ic[1].get("text_excerpt", "")),
            list(enumerate(cases)),
        ))  # sorted by note index
    scores = [g[1] for g in graded]
    mean   = sum(scores) / len(scores)
    worst_k = min(range(len(graded)), key=lambda k: graded[k][1])
    # Surface EVERY note's reasoning to the rewriter, not just the weakest.
    # Multiple notes can miss for different reasons; the rewriter needs the
    # full surface to address them in one shot. The weakest is tagged so
    # the rewriter can prioritise it.
    lines = [f"mean {mean:.2f} over {len(scores)} note(s)"]
    for k, (i, sc, why) in enumerate(graded):
        tag = " [WEAKEST]" if k == worst_k else ""
        lines.append(f"  note{i+1} ({sc:.2f}){tag}: {why}")
    reason = "\n".join(lines)
    return mean, reason


def _rewrite_query(original: str, reason: str) -> str:
    """Generate a HYPOTHETICAL DISCHARGE-NOTE PARAGRAPH (HyDE — Gao et al.
    2022) that, if it existed in the MIMIC-IV corpus, would be the ideal
    retrieval match for this case. The next-round retrieval embeds this
    document directly. Doc-shaped text embeds closer to doc-shaped corpus
    entries than query-shaped text does, so this is more consistent than
    open-ended query rewriting."""
    system = (
        "You generate hypothetical discharge-note paragraphs for a HyDE retrieval "
        "pipeline (Gao et al. 2022) over a FAISS index of MIMIC-IV hospital "
        "DISCHARGE SUMMARIES. Your output is used directly as the next-round "
        "retrieval embedding — so it must READ LIKE A DISCHARGE NOTE, in the "
        "same third-person clinical register the corpus uses (\"Patient is a "
        "[age]M/F with PMH of X who presents with Y...\")."
    )
    prompt = (
        "Generate a hypothetical discharge-note paragraph that, if it existed "
        "in the MIMIC-IV corpus, would be the IDEAL retrieval match for this case.\n\n"
        f"CASE FACTS (this is the ONLY clinical content you may use — do NOT add\n"
        f"symptoms, diagnoses, or findings that are not stated here):\n{original}\n\n"
        f"WHY THE PREVIOUS QUERY UNDER-RETRIEVED (grader reasoning):\n{reason}\n\n"
        "Write ONE dense paragraph (<= 60 words) in the third-person clinical voice "
        "of a MIMIC-IV discharge HPI. Hard rules:\n"
        "- Focus on the CHIEF COMPLAINT and HPI. Mention relevant PMH only if it\n"
        "  is stated in CASE FACTS and clinically informs the presentation.\n"
        "- DO NOT enumerate the full medication list in prose. Mention a class\n"
        "  (e.g. \"on antiplatelet therapy\") only if it is clinically relevant; otherwise omit.\n"
        "- DO NOT include any literal placeholder text like \"[age]\" or \"[sex]\"\n"
        "  or \"X\". If demographics are not stated in CASE FACTS, write the\n"
        "  HPI without naming them (e.g. start with \"Patient presents with...\").\n"
        "- Translate lay wording to standard terms (\"can't breathe\" -> dyspnea;\n"
        "  \"coughed up blood\" -> hemoptysis; \"fell\" -> mechanical fall).\n"
        "- You MAY add synonyms/abbreviations (NSTEMI, COPD exacerbation, PE/DVT)\n"
        "  ONLY for findings already in CASE FACTS.\n"
        "- DO NOT invent symptoms CASE FACTS did not mention.\n"
        "- DO NOT hedge demographics (\"adult/pediatric?\") — omit what is not stated.\n"
        "- Use the grader's weakness hint to broaden coverage of the case facts,\n"
        "  not to invent new ones.\n"
        "Output ONLY the hypothetical HPI paragraph."
    )
    try:
        out = LLM.invoke(
            [SystemMessage(content=system), HumanMessage(content=prompt)]
        ).content.strip().strip('"').strip()
        return out or original
    except Exception:
        return (f"{original}. Discharge summary: primary diagnosis, cardinal "
                "symptoms, abnormal vitals, comorbidities, standard management.")

### 5.3 · Agentic RAG retrieval loop

In [ ]:
def agentic_rag_retrieve(query: str, k: int = 5, max_rewrites: int = 1,
                          threshold: float = 0.5) -> dict:
    """
    Iterative RAG subagent: retrieve -> grade -> rewrite -> retrieve (max 2 rewrites).
    Flags low_confidence if relevance stayed below threshold after all rewrites.
    """
    current_query  = query
    rewrites: list = []
    best = None  # (score, cases, reason, query) - best pass seen so far

    pass_n = 0
    for _ in range(max_rewrites + 1):
        pass_n += 1
        cases         = _faiss_search(current_query, k=k)
        score, reason = _grade_relevance(current_query, cases)
        print(f"  [rag pass {pass_n}] mean {score:.2f} over {len(cases)} note(s)")
        _pfx = f"  [rag pass {pass_n}] query: "
        _ind = " " * len(_pfx)
        # Respect paragraph breaks (\n\n) and heading-on-own-line structure
        # so the CC / HPI sections stay visually distinct in the log.
        _paragraphs = current_query.split("\n\n")
        for _pi, _para in enumerate(_paragraphs):
            if _pi > 0:
                print(_ind)
            _lead = _pfx if _pi == 0 else _ind
            _parts = _para.split("\n", 1)
            if len(_parts) == 2 and _parts[0].endswith(":") and len(_parts[0]) <= 40:
                # Heading on its own line, then body wrapped underneath.
                print(_lead + _parts[0])
                print(textwrap.fill(_parts[1], width=92,
                                    initial_indent=_ind,
                                    subsequent_indent=_ind))
            else:
                print(textwrap.fill(_para.replace("\n", " "), width=92,
                                    initial_indent=_lead,
                                    subsequent_indent=_ind))
        if best is None or score > best[0]:
            best = (score, cases, reason, current_query)
        # Skip rewrite if pass-1 is already within 0.10 of threshold —
        # the rewriter is unreliable in that zone and often regresses.
        _near = threshold - 0.10
        if score >= threshold or score >= _near or len(rewrites) >= max_rewrites:
            if score >= threshold:
                _why = "above threshold"
            elif score >= _near:
                _why = f"close to threshold ({score:.2f} >= {_near:.2f}); rewrite skipped"
            else:
                _why = "max rewrites reached"
            print(f"  [rag pass {pass_n}] kept ({_why})")
            break
        # Surface the grader's most-useful hint — the WEAKEST note's reason.
        # (The summary "mean X over N notes" first line has no signal.)
        _hint = ""
        for _ln in (reason or "").split("\n"):
            if "[WEAKEST]" in _ln:
                _hint = _ln.strip()[:140]
                break
        if not _hint:
            # Fallback: any per-note line (skip the summary line).
            _lines = [l for l in (reason or "").split("\n")[1:] if l.strip()]
            _hint = _lines[0].strip()[:140] if _lines else ""
        print(f"  [rag pass {pass_n}] below threshold ({threshold}) — rewriting")
        if _hint:
            print(f"  [rag pass {pass_n}] grader: {_hint}")
        new_q = _rewrite_query(current_query, reason)
        _pfx = f"  [rag pass {pass_n}] new query: "
        print(textwrap.fill(new_q, width=92,
                            initial_indent=_pfx,
                            subsequent_indent=" " * len(_pfx)))
        rewrites.append({"original": current_query, "rewritten": new_q, "score": score})
        current_query = new_q

    # Return the BEST pass, not the last - rewrites are not monotonic, so a
    # later rewrite can score lower than an earlier one.
    score, cases, reason, current_query = best
    # low_confidence boundary matches Option B's "kept (close to threshold)"
    # decision. If we already accepted the pass because it was within 0.10 of
    # the formal threshold, do NOT also flag it as low-confidence — that
    # contradiction made the orchestrator retry the tool.
    low_confidence = score < (threshold - 0.10)

    if cases:
        _r = f" (kept after {len(rewrites)} rewrite{'s' if len(rewrites)!=1 else ''})" if rewrites else ""
        print(f"  [rag] BEST: {len(cases)} case(s) | mean {score:.2f}{_r}")
        for c in cases:
            _t = " ".join(c.get("text_excerpt", "").split())
            # Extract Chief Complaint and HPI separately so the snippet shows
            # WHY this case is similar, not the boilerplate redaction header.
            _cc = ""
            _m = re.search(r"(?i)chief complaint:\s*(.{1,90}?)(?=\s+(?:major surgical|history of present illness|hpi|past medical|allergies|admission date))", _t)
            if _m: _cc = _m.group(1).strip().rstrip(":")
            _hpi = ""
            _m = re.search(r"(?i)(?:history of present illness|hpi):\s*(.{1,170})", _t)
            if _m: _hpi = _m.group(1).strip()
            print(f"    [{c.get('similarity',0):.2f}] {c.get('id','?')}")
            if _cc:
                print(f"           CC : {_cc}")
            if _hpi:
                print(f"           HPI: {_hpi}...")
            if not _cc and not _hpi:
                # Fallback: post-header snippet if structured fields missing.
                _m = re.search(r"(?i)\b(chief complaint|history of present illness|hpi|service)\b", _t)
                _start = _m.start() if _m else 0
                print(f"           {_t[_start:_start+160]}...")
    return {
        "cases":              cases,
        "relevance_score":    score,
        "relevance_reasoning": reason,
        "query_used":         current_query,
        "rewrites":           rewrites,
        "low_confidence":     low_confidence,
    }

### 5.4 · Build the index

In [ ]:
build_report_index()
print(f"RAG corpus: {FAISS_INDEX.ntotal if FAISS_INDEX else 0:,} vectors in FAISS index.")

## 6 · Nine Tool Implementations

**Assessment** — `assess_case_urgency`, `retrieve_clinical_data`, `draft_treatment_plan` ·
**Safety** — `screen_drug_conflicts`, `assess_allergy_risk`, `render_safety_verdict` ·
**System** — `recall_case_history`, `compile_clinical_note`, `activate_emergency`.

`screen_drug_conflicts` resolves drug CUIs and fetches real DDI data from NLM RxNav
(meds + candidate actions + RAG-extracted drugs); `assess_allergy_risk` pulls
OpenFDA FAERS reaction data — both via the pooled session, then graded by `LLM`.


### 6.0 · Shared LLM→JSON helper

In [ ]:
def _llm_json(llm: Any, system: str, user: str, default: dict) -> dict:
    """Call LLM, strip markdown fences, parse JSON. Returns default on any failure."""
    try:
        resp = llm.invoke([SystemMessage(content=system), HumanMessage(content=user)])
        raw  = re.sub(r"```(?:json)?|```", "", resp.content).strip()
        return json.loads(raw)
    except Exception as e:
        default["_parse_error"] = str(e)
        return default


# ASSESSMENT LAYER

### 6.1 · `assess_case_urgency`

In [ ]:
@tool
def assess_case_urgency(
    symptoms: list,
    vitals: Optional[dict] = None,
    medications: Optional[list] = None,
    allergies: Optional[list] = None,
    missing_data: Optional[list] = None,
    acuity_signals: Optional[list] = None,
) -> dict:
    """
    Triage Agent: assign an emergency-triage tier (critical / urgent / routine)
    from vitals, symptom context, comorbidities, qualitative acuity cues and
    missing-data patterns. ALWAYS call this first before any other tool.
    Pass `acuity_signals` (the case's qualitative red-flag cues) whenever present.
    Returns: urgency, reasoning, recommended_path (escalate/continue/clarify),
    confidence (0-1), missing_critical_data, red_flags.
    """
    vitals         = vitals or {}
    medications    = medications or []
    allergies      = allergies or []
    missing_data   = missing_data or []
    acuity_signals = acuity_signals or []

    system = """You are an emergency-medicine triage specialist. Assign ONE tier
that mirrors how an ED triage nurse assigns ESI acuity. Reason holistically
about the whole presentation and its trajectory, not isolated thresholds.

HARDEST PRIORITY — evaluate FIRST, before any rule below.
When an override matches, the downgrade rules below ("between-tiers lower",
"missing-data lower confidence") DO NOT APPLY.

  STROKE: facial droop/numbness OR unilateral arm/leg weakness OR
  dysarthria/aphasia/"speech off"/"word-finding difficulty"/
  "thoughts slide away"/"not making sense"/"saying things that don't fit"
  — AND acute onset (today, wake-up, or "don't know when") → **critical**,
  confidence ≥ 0.70. Code Stroke. Missing vitals or patient coherence do NOT
  downgrade. Wake-up counts. Lay-language aphasia counts as aphasia.

  STEMI/ACS: acute crushing/pressure chest pain + (radiation arm/jaw OR
  diaphoresis OR dyspnoea OR syncope) → **critical**, ≥ 0.70. Coherence
  does NOT downgrade.

  ANAPHYLAXIS: airway swelling/stridor/hoarseness OR hives+dyspnoea OR
  hypotension post-exposure → **critical**.

  SPINAL CORD / SEVERE TRAUMA: post-fall, post-MVC, or other trauma WITH
  bilateral neurologic deficit (bilateral numbness/weakness) OR new
  paraplegia/quadriplegia OR loss of bowel/bladder control OR significant
  neck/back pain after trauma → **critical**, confidence ≥ 0.70. Cord
  injury cannot be ruled out without imaging; treat as time-critical.

  ACUTE SEVERE AMS: acute confusion / agitation / lethargy that family
  describes as "not making sense", "found unresponsive", "couldn't wake
  up", or rapidly worsening alteration of consciousness without obvious
  reversible cause → **critical**, confidence ≥ 0.70.

If no override matches, apply the standard rubric below.

TIER DEFINITIONS (ESI-aligned)
- "critical" (ESI-1): the patient needs an IMMEDIATE life-saving intervention
  in the next few minutes — peri-arrest or actively decompensating. Examples:
  unresponsive / rapidly declining consciousness, apnoea or agonal/again
  failing breathing, airway compromise, SpO2 < 85 despite oxygen, profound
  shock (SBP < 80, cold/mottled/no perfusion), active major haemorrhage with
  instability, status / ongoing seizure, unstable arrhythmia with
  hypoperfusion, anaphylaxis with airway or circulatory collapse. The decisive
  test: "without intervention within minutes this patient arrests."
  recommended_path = "escalate".
- "urgent" (ESI-2): high-risk, time-sensitive, must be seen quickly, but
  currently MAINTAINING airway/breathing/circulation. This includes serious,
  even potentially life-threatening diagnoses that are NOT peri-arrest:
  chest pain suspicious for ACS but haemodynamically stable and speaking;
  new focal neuro deficit / stroke window but protecting airway; GI bleed
  with stable vitals; severe pain; sepsis without shock; acute confusion that
  is stable; a seizure that has STOPPED or hypoglycaemia already treated and
  the patient now responsive. recommended_path = "continue".
- "routine" (ESI-3+): physiologically stable, subacute or chronic, coherent
  history, no objective acute red flag; work-up can be scheduled.
  recommended_path = "continue".

DECISIVE RULE — critical vs urgent
The split is whether an IMMEDIATE (minutes) life-saving intervention is
required, NOT how serious or frightening the diagnosis is. A serious or even
life-threatening condition that is currently compensated or has been
stabilised is "urgent", not "critical". Recent stabilising treatment
(glucose given and patient now responsive; seizure now resolved; bleeding
that has slowed with stable vitals) DOWNGRADES critical -> urgent unless
active decompensation persists. Do not escalate to critical merely because
the condition "could deteriorate".

MISSING-DATA RULE
Absence of numeric vitals is NOT by itself a reason to escalate. When numbers
are missing, triage from the qualitative cues and symptoms; record what is
missing in "missing_critical_data" and LOWER "confidence". Do not inflate the
tier because data is incomplete.

OBJECTIVE vs SUBJECTIVE
Alarmed language ("please help me", "I'm scared") is NOT a triage signal.
Use objective findings + the HARDEST PRIORITY overrides above.

GROUND-TRUTH NOTE
The reference label is a real ED triage acuity influenced by measured vitals
and context not always present in a lay narrative. Assign the
clinically-defensible tier from what IS stated. Do not inflate a
serious-but-stable presentation to critical, and do not deflate an objective
peri-arrest because the patient sounds calm. When genuinely between two
tiers, choose the lower one and lower confidence.

CALIBRATION EXAMPLES (lay-narrative style)
- "going dark, can't stay awake" -> critical (peri-arrest).
- "gasping, lips blue" -> critical.
- "left side weak, face funny, speech off, started overnight" -> critical
  (FAST stroke; coherence + missing vitals do NOT downgrade).
- "crushing chest pressure radiating to arm, sweating, dyspnoea" -> critical (STEMI).
- "crushing chest pressure, no radiation/diaphoresis/dyspnoea" -> urgent (chest pain w/o ACS features).
- "seizure stopped, glucose given, awake now" -> urgent (stabilised, NOT critical).
- "3 days N/V/D, otherwise okay" -> routine.

Return JSON ONLY (no markdown, no prose outside the object):
{
  "urgency": "critical"|"urgent"|"routine",
  "reasoning": "<concise clinical rationale, 2-3 sentences; name the single
                fact that fixed the tier>",
  "recommended_path": "escalate"|"continue"|"clarify",
  "confidence": <0.0-1.0>,
  "missing_critical_data": ["<item>"],
  "red_flags": ["<OBJECTIVE peri-arrest sign only>"]
}"""
    user = (
        f"Symptoms: {symptoms}\n"
        f"Vitals: {vitals}\n"
        f"Acuity signals (patient's qualitative red-flag cues): {acuity_signals}\n"
        f"Medications: {medications}\n"
        f"Allergies: {allergies}\n"
        f"Missing data: {missing_data}"
    )

    _TIERS   = ("critical", "urgent", "routine")
    _bad     = {"urgency": "__unparsed__"}
    out = _llm_json(LLM_THINK, system, user, dict(_bad))
    if out.get("urgency") not in _TIERS:                       # one stricter retry
        out = _llm_json(
            LLM_THINK,
            system + "\n\nReturn ONLY the JSON object — no other text.",
            user, dict(_bad),
        )
    if out.get("urgency") not in _TIERS:                       # flagged fallback
        out = {
            "urgency": "urgent",
            "reasoning": "Triage model output could not be parsed; defaulting "
                         "conservatively pending clinician review.",
            "recommended_path": "clarify",
            "confidence": 0.3,
            "missing_critical_data": ["reliable triage assessment"],
            "red_flags": [],
            "_triage_unparsed": True,
        }

    urgency    = out["urgency"]
    confidence = float(out.get("confidence", 0.7))
    return AgentHandoff(
        tool_name="assess_case_urgency",
        input_received={"symptoms": symptoms, "vitals": vitals,
                        "acuity_signals": acuity_signals},
        reasoning_trace=out.get("reasoning", ""),
        output=out,
        confidence=confidence,
        caveats=out.get("missing_critical_data", []),
        suggested_next_tools=["activate_emergency"] if urgency == "critical" else ["retrieve_clinical_data"],
    ).to_dict()

### 6.2 · `retrieve_clinical_data`

In [ ]:
def _infer_chief_complaint(query: str) -> str:
    """Extract a discharge-note-style Chief Complaint (<= 12 words, clinical
    register) from a longer clinical query. Mirrors MIMIC-IV's CC field so
    the dense embedder can match CC-to-CC, not just HPI-to-HPI."""
    if not query or not query.strip():
        return ""
    system = (
        "You extract Chief Complaint lines for clinical retrieval against "
        "MIMIC-IV discharge notes. Output a single concise CC line in the "
        "style real ED triage and discharge notes use:\n"
        "- 12 words or fewer.\n"
        "- Clinical terms only (no lay language).\n"
        "- Examples: \"Hemoptysis\", \"Right-sided weakness\", \"Chest pain\", "
        "\"Acute onset dizziness, concern for stroke\", \"Mechanical fall with hip pain\".\n"
        "- Do NOT prepend \"Chief Complaint:\" — just the content.\n"
        "- Do NOT invent symptoms not in the input."
    )
    prompt = f"Clinical context:\n{query}\n\nOutput ONLY the Chief Complaint line."
    try:
        raw = LLM.invoke([SystemMessage(content=system),
                          HumanMessage(content=prompt)]).content
        cc = raw.strip().strip('"').strip("'")
        cc = re.sub(r"(?i)^chief complaint:\s*", "", cc).strip()
        return cc[:120]
    except Exception:
        return ""


@tool
def retrieve_clinical_data(clinical_query: str) -> dict:
    """
    Agentic RAG Subagent: iterative retrieval with relevance grading and query rewriting
    (up to 2 rewrites) over indexed EHR corpus.
    Provide a descriptive clinical summary as the query.
    Returns: retrieved cases, relevance score, query rewrites performed, low_confidence flag.
    """
    # Infer a CC-style anchor and prepend it so the dense embedder matches
    # MIMIC-IV's top-of-note Chief Complaint field directly. The augmented
    # query uses MIMIC's actual layout — CC under its own heading on its
    # own line, with an explicit "History of Present Illness:" heading on
    # the body — which adds structural alignment on top of topical.
    _cc = _infer_chief_complaint(clinical_query)
    if _cc:
        print(f"  [rag] inferred CC: {_cc}")
        _augmented_query = (
            f"Chief Complaint:\n{_cc}\n\n"
            f"History of Present Illness:\n{clinical_query}"
        )
    else:
        _augmented_query = clinical_query
    rag  = agentic_rag_retrieve(_augmented_query, k=5)
    rag["chief_complaint"] = _cc
    rag["drug_patterns"] = _extract_drug_patterns(rag.get("cases", []))
    cav  = []
    if rag["low_confidence"]:
        cav.append("Retrieval relevance is low even after refinement — "
                   "DO NOT retry this tool; proceed with current evidence.")
    if rag["rewrites"]:
        cav.append(f"Query rewritten {len(rag['rewrites'])} time(s).")
    return AgentHandoff(
        tool_name="retrieve_clinical_data",
        input_received={"clinical_query": clinical_query},
        reasoning_trace=rag["relevance_reasoning"],
        output=rag,
        confidence=min(rag["relevance_score"] + 0.15, 1.0),
        caveats=cav,
        suggested_next_tools=["draft_treatment_plan"],
    ).to_dict()

### 6.3 · `draft_treatment_plan`

In [ ]:
@tool
def draft_treatment_plan(
    symptoms: list,
    retrieved_evidence: str,
    current_medications: list,
    documented_allergies: list,
    rag_drug_patterns: Optional[list] = None,
    critique_feedback: Optional[str] = None,
) -> dict:
    """
    Action Agent: generate 3-5 candidate treatment interventions grounded in retrieved evidence.
    States explicit assumptions for missing data; indicates uncertainty level per action.
    Does NOT issue final decisions — produces candidates for downstream safety verification.
    """
    system = """You are a conservative clinical decision-support assistant.
Generate 3-5 candidate treatment interventions. For each: state rationale, uncertainty level
(low/medium/high), assumptions made due to missing data, and evidence citations.
Use the provided "drugs used in similar cases" list as a CANDIDATE POOL — these are drugs
real clinicians prescribed for patients with comparable presentations. Prefer drugs from
this pool when clinically appropriate, and cite the case_ids in evidence_citations. If you
propose a drug NOT in the pool, briefly justify the choice in the action's rationale.
Do NOT prescribe — produce candidates for safety review.
Return JSON only:
{
  "candidate_actions": [
    {"action": "<intervention>", "rationale": "<why>", "uncertainty": "low"|"medium"|"high",
     "assumptions": ["<assumption>"], "evidence_citations": ["<ref>"]}
  ],
  "general_assumptions": ["<assumption>"],
  "overall_confidence": <0.0-1.0>
}"""
    _fb_block = ""
    if critique_feedback:
        _fb_block = (
            "\n\n[CRITIQUE FEEDBACK on the previous draft]\n"
            f"{critique_feedback}\n"
            "Fix the issue above SPECIFICALLY in this new draft. "
            "Do not reproduce the same problem. Re-check every drug "
            "name spelling against the standard generic spelling. "
            "Keep all other clinically-sound parts of the plan.")
    user = (
        f"Symptoms: {symptoms}\n"
        f"Current medications: {current_medications}\n"
        f"Documented allergies: {documented_allergies}\n"
        f"Drugs used in similar cases (use as candidate-drug pool — cite case_ids in evidence_citations):\n"
        f"{json.dumps(rag_drug_patterns or [], indent=2)}\n\n"
        f"Full retrieved evidence:\n{retrieved_evidence}"
        f"{_fb_block}"
    )
    out = _llm_json(LLM, system, user, {
        "candidate_actions": [{"action": "Diagnostic workup", "rationale": "Insufficient data",
                               "uncertainty": "high", "assumptions": [], "evidence_citations": []}],
        "general_assumptions": [], "overall_confidence": 0.4
    })
    n_acts = len(out.get("candidate_actions", []))
    return AgentHandoff(
        tool_name="draft_treatment_plan",
        input_received={"symptoms": symptoms, "medications": current_medications,
                        "allergies": documented_allergies},
        reasoning_trace=f"Generated {n_acts} candidate action(s). Confidence: {out.get('overall_confidence', 0.5)}",
        output=out,
        confidence=float(out.get("overall_confidence", 0.6)),
        caveats=out.get("general_assumptions", [])[:3],
        suggested_next_tools=["screen_drug_conflicts", "assess_allergy_risk"],
    ).to_dict()


# SAFETY LAYER

### 6.4 · `screen_drug_conflicts` — RxNav DDI

In [ ]:
def _extract_drugs_from_rag_cases(cases: list[dict], max_cases: int = 5) -> list[str]:
    if not cases:
        return []
    combined = "\n---\n".join(
        f"Case {i+1}:\n{c.get('text_excerpt','')}"
        for i, c in enumerate(cases[:max_cases]) if c.get("text_excerpt")
    )
    if not combined:
        return []
    prompt = (
        "Extract all drug and medication names from these clinical note excerpts.\n"
        "Include brand names, generics. Exclude labs, devices, procedures.\n\n"
        f"{combined}\n\n"
        'Return JSON only: {"drugs": ["<drug name>", ...]}'
    )
    try:
        raw = LLM.invoke([HumanMessage(content=prompt)]).content
        raw = re.sub(r"```(?:json)?|```", "", raw).strip()
        drugs = [d.strip().lower() for d in json.loads(raw).get("drugs", [])
                 if isinstance(d, str) and d.strip()]
        return list(dict.fromkeys(drugs))
    except Exception:
        return []


def _extract_drug_patterns(cases: list[dict], max_cases: int = 5) -> list[dict]:
    """Aggregate drug mentions across retrieved cases. Returns list of
    {drug, count, case_ids} sorted by frequency. Used by draft_treatment_plan
    as a structured candidate-drug pool (NOT for DDI screening — those drugs
    do not belong to this patient)."""
    if not cases:
        return []
    counts: dict[str, list[str]] = {}
    for c in cases[:max_cases]:
        cid = c.get("id", "?")
        names = _extract_drugs_from_rag_cases([c])
        for n in names:
            counts.setdefault(n, []).append(cid)
    patterns = [{"drug": d, "count": len(cids), "case_ids": cids}
                for d, cids in counts.items()]
    patterns.sort(key=lambda p: -p["count"])
    return patterns[:20]


def _get_rxcui(drug_name: str) -> str:
    """Resolve a drug name to an RxNorm CUI.
    Falls back through three layers when the LLM emits an awkward name:
      1. RxNav exact match (rxcui.json?search=1)
      2. RxNav approximate-term fuzzy match (approximateTerm.json)
      3. OpenFDA Drug Label → generic_name → RxNav exact match
    Returns "" if all three fail.
    """
    n = drug_name.strip()
    if not n:
        return ""
    # (1) Exact match
    try:
        r = _HTTP.get(
            "https://rxnav.nlm.nih.gov/REST/rxcui.json",
            params={"name": n, "search": "1"},
            timeout=6,
        )
        if r.status_code == 200:
            ids = r.json().get("idGroup", {}).get("rxnormId", [])
            if ids:
                return ids[0]
    except Exception:
        pass
    # (2) RxNav fuzzy approximateTerm
    try:
        r = _HTTP.get(
            "https://rxnav.nlm.nih.gov/REST/approximateTerm.json",
            params={"term": n, "maxEntries": 1},
            timeout=6,
        )
        if r.status_code == 200:
            cands = r.json().get("approximateGroup", {}).get("candidate", [])
            if cands and cands[0].get("rxcui"):
                return cands[0]["rxcui"]
    except Exception:
        pass
    # (3) OpenFDA Drug Label → generic_name → RxNav exact
    try:
        r = _HTTP.get(
            "https://api.fda.gov/drug/label.json",
            params={"search": f"openfda.brand_name:\"{n}\" "
                              f"openfda.generic_name:\"{n}\"",
                    "limit": 1},
            timeout=6,
        )
        if r.status_code == 200:
            results = r.json().get("results", [])
            if results:
                gn_list = (results[0].get("openfda", {}) or {}).get("generic_name", [])
                if gn_list:
                    canonical = gn_list[0]
                    rr = _HTTP.get(
                        "https://rxnav.nlm.nih.gov/REST/rxcui.json",
                        params={"name": canonical, "search": "1"},
                        timeout=6,
                    )
                    if rr.status_code == 200:
                        ids = rr.json().get("idGroup", {}).get("rxnormId", [])
                        if ids:
                            return ids[0]
    except Exception:
        pass
    return ""


def _fetch_ddi_rxnav(drug_a: str, drug_b: str, cui_a: str, cui_b: str) -> tuple[str, str]:
    if not cui_a or not cui_b:
        return ("", "")
    try:
        r = _HTTP.get(
            "https://rxnav.nlm.nih.gov/REST/interaction/list.json",
            params={"rxcuis": f"{cui_a}+{cui_b}"},
            timeout=6,
        )
        if r.status_code != 200:
            return ("", "")
        hits = []
        for group in r.json().get("fullInteractionTypeGroup", []):
            for itype in group.get("fullInteractionType", []):
                for pair in itype.get("interactionPair", []):
                    sev  = pair.get("severity", "").upper() or "UNKNOWN"
                    desc = pair.get("description", "")
                    hits.append(f"{sev}: {desc}")
        if not hits:
            return ("", "")
        src = (
            f"https://rxnav.nlm.nih.gov/REST/interaction/list.json"
            f"?rxcuis={cui_a}+{cui_b}"
        )
        summary = f"RxNav DDI [{drug_a} x {drug_b}]: " + " | ".join(hits[:3])
        print(f"  [d3] web: {summary[:100]}")
        print(f"  [d3] src: {src[:110]}")
        return (summary, src)
    except Exception:
        return ("", "")


@tool
def screen_drug_conflicts(
    candidate_actions: list,
    current_medications: list,
    rag_cases: Optional[list] = None,
) -> dict:
    """
    D3 Drug-Drug Interaction classifier: screen candidate actions for
    pharmacokinetic and pharmacodynamic drug-drug interactions.
    Call this when candidate actions include prescription or OTC medications.
    Returns: blocked_actions, caution_actions, risk_factors, uncertainty_flags.
    """
    rag_cases = rag_cases or []

    # 1. Normalise current med names
    med_names = [
        (m.get("name", "") if isinstance(m, dict) else str(m)).strip().lower()
        for m in current_medications
    ]
    med_names = [m for m in med_names if m]

    # 2. Candidate action strings — extract drug-name candidates only.
    # Reject action descriptions ("diagnostic workup", "neurology consultation")
    # to keep the RxNav fuzzy matcher from inventing spurious DDI findings.
    _ACTION_STOPWORDS = (
        "workup", "evaluation", "consultation", "management", "monitoring",
        "observation", "assessment", "review", "consideration", "imaging",
        "labs", "discharge", "admission", "transfer", "referral",
        "examination", "telemetry", "stabilization", "supportive care",
        "fluid resuscitation", "rehabilitation", "follow-up", "education",
        "counseling", "screen for", "rule out", "neurologic exam",
    )
    cand_names_raw = [
        (a.get("action", str(a)) if isinstance(a, dict) else str(a)).strip().lower()
        for a in candidate_actions
    ]
    cand_names = []
    for _name in cand_names_raw:
        if not _name:
            continue
        # Drop if it contains a non-drug action verb
        if any(sw in _name for sw in _ACTION_STOPWORDS):
            continue
        # Drop overly long strings (real drug names + dose are 1-3 words;
        # 6+ words is almost always a description, not a drug)
        if len(_name.split()) >= 6:
            continue
        cand_names.append(_name)

    # 3. Drugs from similar RAG cases
    rag_drug_names = _extract_drugs_from_rag_cases(rag_cases)
    if rag_drug_names:
        print(f"  [d3] rag drugs extracted: {rag_drug_names[:6]}")

    # RAG-extracted drugs are OTHER patients' meds — do NOT include in the
    # patient-meds premise for DDI lookup. They stay visible to the LLM
    # grader below as 'context' (drugs commonly used in similar cases).
    all_drug_names = list(dict.fromkeys(med_names + cand_names))

    # 4. Resolve CUIs in parallel then fetch DDI data for all unique pairs
    ddi_texts:   list[str] = []
    ddi_sources: list[str] = []

    if all_drug_names:
        with ThreadPoolExecutor(max_workers=8) as exe:
            cui_futures = {n: exe.submit(_get_rxcui, n) for n in all_drug_names}
        resolved = {n: f.result() for n, f in cui_futures.items() if f.result()}
        if resolved:
            print(f"  [d3] resolved {len(resolved)}/{len(all_drug_names)} CUIs: "
                  f"{list(resolved.keys())[:5]}")

        drug_list = list(resolved.keys())
        pairs = [
            (drug_list[i], drug_list[j])
            for i in range(len(drug_list))
            for j in range(i + 1, len(drug_list))
        ][:20]

        with ThreadPoolExecutor(max_workers=6) as exe:
            pair_futs = {
                (a, b): exe.submit(_fetch_ddi_rxnav, a, b, resolved[a], resolved[b])
                for a, b in pairs
            }
        for (a, b), fut in pair_futs.items():
            text, src = fut.result()
            if text:
                ddi_texts.append(text)
                ddi_sources.append(src)

    ddi_str = "\n".join(ddi_texts) if ddi_texts else "No RxNav DDI data retrieved."

    system = """You are a clinical pharmacologist specialising in drug-drug interactions.
Evaluate ONLY drug-drug interactions between candidate interventions and current medications.
You are given real-world DDI data from NLM RxNav — use it as ground truth where available.
1. Pharmacokinetic interactions (absorption, distribution, metabolism, excretion)
2. Pharmacodynamic interactions (additive/synergistic/antagonistic effects)
3. Missing-data uncertainty (unknown renal/hepatic function, weight, etc.)
Do NOT flag drug-disease contraindications here.
Be conservative — flag anything potentially unsafe.
Severity levels MUST be exactly: "MAJOR", "MODERATE", or "MINOR" (uppercase).
Return JSON only:
{
  "blocked_actions": [],
  "caution_actions": [{"action": "<action>", "concern": "<concern>", "severity": "MAJOR"|"MODERATE"|"MINOR", "management": "<mgmt>"}],
  "risk_factors": ["<risk>"],
  "uncertainty_flags": ["<missing data item>"],
  "overall_safety_confidence": <0.0-1.0>
}"""
    user = (
        f"Candidate interventions:\n{json.dumps(candidate_actions, indent=2)}\n\n"
        f"Current medications:\n{json.dumps(current_medications, indent=2)}\n\n"
        f"Drugs used in similar RAG cases:\n{json.dumps(rag_drug_names, indent=2)}\n\n"
        f"Real-world DDI data (NLM RxNav):\n{ddi_str}"
    )
    out = _llm_json(LLM, system, user, {
        "blocked_actions": [], "caution_actions": [], "risk_factors": [],
        "uncertainty_flags": ["D3 screening unavailable"],
        "overall_safety_confidence": 0.5
    })
    n_blocked = len(out.get("blocked_actions", []))
    n_caution = len(out.get("caution_actions", []))
    for a in out.get("blocked_actions", []):
        print(f"  [d3] BLOCKED: {str(a)[:70]}")
    for a in out.get("caution_actions", []):
        sev = a.get("severity", "?").upper() if isinstance(a, dict) else "?"
        act = a.get("action", str(a))[:45] if isinstance(a, dict) else str(a)[:45]
        con = a.get("concern", "")[:50] if isinstance(a, dict) else ""
        print(f"  [d3] {sev}: {act} ? {con}")
    sev_summary = ", ".join(
        f"{a.get('action','')[:20]}({a.get('severity','?')})"
        for a in out.get("caution_actions", []) if isinstance(a, dict)
    )
    return AgentHandoff(
        tool_name="screen_drug_conflicts",
        input_received={"candidate_actions": candidate_actions,
                        "medications": current_medications,
                        "rag_cases_count": len(rag_cases)},
        reasoning_trace=f"D3: {n_blocked} blocked, {n_caution} cautions. {sev_summary}",
        output={**out, "ddi_sources": ddi_sources},
        confidence=float(out.get("overall_safety_confidence", 0.7)),
        caveats=out.get("uncertainty_flags", []),
        suggested_next_tools=["render_safety_verdict"],
    ).to_dict()

### 6.5 · `assess_allergy_risk` — OpenFDA

In [ ]:
def _fetch_allergy_guidelines(allergen: str) -> tuple[str, str]:
    """Query OpenFDA FAERS for top adverse reactions reported for this allergen."""
    try:
        clean  = allergen.lower().strip()
        url    = "https://api.fda.gov/drug/event.json"
        params = {
            "search": f'patient.drug.medicinalproduct:"{clean}"',
            "count":  "patient.reaction.reactionmeddrapt.exact",
            "limit":  8,
        }
        r = _HTTP.get(url, params=params, timeout=6)
        if r.status_code == 200:
            results   = r.json().get("results", [])
            reactions = [item["term"] for item in results[:8]]
            src_url   = f"https://api.fda.gov/drug/event.json?search=patient.drug.medicinalproduct:{clean}&count=patient.reaction.reactionmeddrapt.exact"
            return (f"FDA FAERS ? top reactions for {allergen}: {chr(44).join(reactions)}", src_url)
    except Exception:
        pass
    return ("", "")


@tool
def assess_allergy_risk(
    candidate_actions: list,
    documented_allergies: list
) -> dict:
    """
    Allergy Agent: assess cross-reactivity risk for candidate actions against documented allergies.
    Evaluates structural similarity, reaction severity, and identifies safer alternative classes.
    Call this when the patient has documented allergies.
    Returns: high_risk_actions, moderate_risk_actions, low_risk_actions, safer_alternatives.
    """
    system = """You are a clinical allergist and immunologist. Classify EVERY
candidate action into exactly one of three buckets based on cross-reactivity
with the documented allergies.

CLASSIFICATION RULES (apply per action):

  high_risk_actions — direct match or established high-rate cross-reactivity
    (e.g., penicillin allergy + amoxicillin/ampicillin/piperacillin;
    sulfa allergy + Bactrim/sulfamethoxazole; NSAID allergy + ibuprofen;
    documented anaphylaxis to the drug class itself). MUST cite the drug
    class overlap in the reason. Use only when reaction is plausible.

  moderate_risk_actions — meaningful but lower-rate cross-reactivity within
    a related drug class (e.g., penicillin allergy + cephalosporins ~2%;
    sulfa allergy + non-antibiotic sulfonamides debated). MUST name the
    specific shared class or structural feature in the reason.

  low_risk_actions — NO plausible pharmacologic class overlap between the
    action and any documented allergen. This is the DEFAULT — assign here
    if you cannot articulate a specific class overlap. Examples that ARE
    low risk: IV crystalloid/electrolytes vs any drug allergy; ondansetron
    vs sulfa/penicillin/macrolide allergies; loperamide vs antibiotic
    allergies; food allergies (shellfish, tomatoes, nuts) vs ANY drug;
    insect-sting allergies vs ANY drug. Output just the action string.

HARD CONSTRAINTS:
- Every candidate action appears in EXACTLY ONE bucket (no duplicates,
  no omissions). high + moderate + low = all candidate actions.
- Do NOT flag based on quantity of allergies. A patient with 8 unrelated
  allergens gets the same per-action classification as a patient with 1
  allergen, for an action that has no class overlap with any of them.
- Do NOT flag IV fluids, electrolytes, oxygen, monitoring, imaging, or
  supportive actions unless the documented allergen is to that specific
  agent (e.g., iodine allergy + IV iodinated contrast).
- If a food/insect-sting allergy is the only entry, drug actions are all
  low_risk unless the proposed drug literally contains the food protein.

CONFIDENCE:
- overall_allergy_confidence ≥ 0.85 when high+moderate are empty.
- ≥ 0.70 when reactions are documented but class overlap is well-known.
- < 0.50 only when allergen identity cannot be verified at all.

Return JSON only:
{
  "high_risk_actions": [{"action": "<action>", "reason": "<class overlap>", "cross_reactivity_rate": "<rate>"}],
  "moderate_risk_actions": [{"action": "<action>", "reason": "<class overlap>", "alternative": "<safer option>"}],
  "low_risk_actions": ["<action>"],
  "safer_alternatives": [{"for": "<action>", "alternative": "<drug/class>", "rationale": "<why>"}],
  "overall_allergy_confidence": <0.0-1.0>
}"""
    guideline_texts = []
    guideline_urls  = []
    for allergy in documented_allergies:
        allergen = allergy.get("allergen", str(allergy)) if isinstance(allergy, dict) else str(allergy)
        text, url = _fetch_allergy_guidelines(allergen)
        if text:
            guideline_texts.append(text)
            guideline_urls.append(url)
            print(f"  [allergy] web: {text[:80]}")
            print(f"  [allergy] src: {url[:90]}")
    guidelines_str = chr(10).join(guideline_texts) if guideline_texts else "No web data retrieved."
    user = (
        f"Candidate interventions:\n{json.dumps(candidate_actions, indent=2)}\n\n"
        f"Documented allergies:\n{json.dumps(documented_allergies, indent=2)}\n\n"
        f"Real-world FDA adverse reaction data (OpenFDA FAERS):\n{guidelines_str}"
    )
    out = _llm_json(LLM, system, user, {
        "high_risk_actions": [], "moderate_risk_actions": [],
        "low_risk_actions": [], "safer_alternatives": [],
        "overall_allergy_confidence": 0.6
    })
    n_hi = len(out.get("high_risk_actions", []))
    n_mo = len(out.get("moderate_risk_actions", []))
    return AgentHandoff(
        tool_name="assess_allergy_risk",
        input_received={"candidate_actions": candidate_actions,
                        "allergies": documented_allergies},
        reasoning_trace=f"Allergy: {n_hi} high-risk, {n_mo} moderate-risk.",
        output={**out, "guideline_sources": guideline_urls},
        confidence=float(out.get("overall_allergy_confidence", 0.7)),
        caveats=([a["action"] for a in out.get("high_risk_actions", [])]
                 + (["Sources: " + ", ".join(guideline_urls[:2])] if guideline_urls else [])),
        suggested_next_tools=["render_safety_verdict"],
    ).to_dict()

### 6.6 · `render_safety_verdict`

In [ ]:
@tool
def render_safety_verdict(
    d3_output: dict,
    allergy_output: dict,
    urgency: str,
    candidate_actions: list
) -> dict:
    """
    Safety Policy Agent: aggregate D3 and allergy evidence into a structured safety verdict.
    Conservative default: hold_pending_clarification when critical safety data is absent.
    Returns: recommendation_class (ACC/AHA: Class_I|IIa|IIb|III_NB|III_Harm),
    order_disposition (ASHP/JC: approve_as_written|approve_with_modification|
    hold_pending_clarification|reject), approved_actions, excluded_actions,
    required_monitoring, stop_conditions, follow_up_actions.
    """
    system = """You are a clinical safety policy agent. Aggregate drug-safety
and allergy evidence into a structured verdict grounded in two established
clinical-decision frameworks:

  recommendation_class — ACC/AHA Class of Recommendation
    (Halperin et al. 2016, Circulation 133:1426)
      "Class_I"        : Strong — benefit >>> risk. SHOULD be performed.
      "Class_IIa"      : Moderate — benefit >> risk. IS REASONABLE to perform.
      "Class_IIb"      : Weak — benefit ≥ risk. MAY BE CONSIDERED.
      "Class_III_NB"   : No benefit. SHOULD NOT be performed (not useful).
      "Class_III_Harm" : Causes harm. MUST NOT be performed.

  order_disposition — Pharmacist drug-order verification
    (ASHP 2016, Am J Health Syst Pharm 73:410; Joint Commission MM.04.01.01)
      "approve_as_written"         : execute the order as drafted.
      "approve_with_modification"  : execute with dose/route/monitoring change.
      "hold_pending_clarification" : do NOT execute; need additional data.
      "reject"                     : contraindicated; cancel the order.

Decision rules (apply in order):
  1. Any d3 blocked action or high-risk allergy action → Class_III_Harm, reject.
  2. urgency=critical AND no absolute contraindication → Class_I or Class_IIa,
     approve_as_written (or approve_with_modification if monitoring needed).
  3. Critical safety data absent AND urgency not extreme →
     hold_pending_clarification, recommendation_class reflects what would be
     done once data arrives (typically Class_IIa or IIb).
  4. When in doubt → hold_pending_clarification (conservative default).

The `verdict` field is a TIGHT STRUCTURED LABEL ONLY (e.g. "Class_IIa — approve_with_modification"). Put all reasoning, qualifications, and clinical caveats in `verdict_reasoning`, NEVER in `verdict`.

Cite the framework in verdict_reasoning, e.g.:
  "Class IIa (LoE C-LD) — reasonable to initiate dual antiplatelet for
   non-cardioembolic stroke. Order disposition: approve_with_modification
   (renal-dose check required)."

Return JSON only:
{
  "recommendation_class": "Class_I"|"Class_IIa"|"Class_IIb"|"Class_III_NB"|"Class_III_Harm",
  "order_disposition":    "approve_as_written"|"approve_with_modification"|"hold_pending_clarification"|"reject",
  "verdict":              "<Class_I|Class_IIa|Class_IIb|Class_III_NB|Class_III_Harm> — <approve_as_written|approve_with_modification|hold_pending_clarification|reject>",  // EXACTLY THIS FORMAT, NO ADDITIONAL TEXT
  "approved_actions":     ["<action>"],
  "excluded_actions":     [{"action": "<action>", "reason": "<why>"}],
  "required_monitoring":  ["<parameter>"],
  "stop_conditions":      ["<condition>"],
  "follow_up_actions":    ["<action>"],
  "verdict_reasoning":    "<2-3 sentences citing ACC/AHA class + pharmacist disposition>",
  "verdict_confidence":   <0.0-1.0>
}"""
    user = (
        f"Urgency: {urgency}\n"
        f"D3 results:\n{json.dumps(d3_output, indent=2)}\n\n"
        f"Allergy results:\n{json.dumps(allergy_output, indent=2)}\n\n"
        f"Candidate actions:\n{json.dumps(candidate_actions, indent=2)}"
    )
    out = _llm_json(LLM, system, user, {
        "recommendation_class": "Class_IIb",
        "order_disposition":    "hold_pending_clarification",
        "verdict":              "Class_IIb — hold_pending_clarification",
        "approved_actions": [], "excluded_actions": [],
        "required_monitoring": [], "stop_conditions": [],
        "follow_up_actions": [], "verdict_reasoning": "Safety evaluation unavailable.",
        "verdict_confidence": 0.4
    })
    return AgentHandoff(
        tool_name="render_safety_verdict",
        input_received={"urgency": urgency},
        reasoning_trace=out.get("verdict_reasoning", ""),
        output=out,
        confidence=float(out.get("verdict_confidence", 0.7)),
        caveats=[f"Excluded: {e['action']}" for e in out.get("excluded_actions", [])],
        suggested_next_tools=["compile_clinical_note"],
    ).to_dict()


# SYSTEM LAYER

### 6.7 · `recall_case_history`

In [ ]:
@tool
def recall_case_history(case_summary: str) -> dict:
    """
    Query Decision Memory for similar past decisions and documented outcomes.
    Unlike retrieve_clinical_data, this retrieves past decision contexts — how similar
    safety flag patterns were handled — not clinical narratives.
    Provide a 1-2 sentence case summary as input.
    """
    past = DECISION_MEMORY.query(case_summary, k=3)
    if not past:
        return AgentHandoff(
            tool_name="recall_case_history",
            input_received={"case_summary": case_summary},
            reasoning_trace="Decision Memory is empty — cold start.",
            output={"past_decisions": [], "count": 0},
            confidence=1.0,
            caveats=["No prior decisions in memory."],
            suggested_next_tools=["retrieve_clinical_data"],
        ).to_dict()
    return AgentHandoff(
        tool_name="recall_case_history",
        input_received={"case_summary": case_summary},
        reasoning_trace=f"Retrieved {len(past)} similar past decisions from Decision Memory.",
        output={"past_decisions": past, "count": len(past)},
        confidence=0.8,
        caveats=[],
        suggested_next_tools=["retrieve_clinical_data"],
    ).to_dict()

### 6.8 · `compile_clinical_note`

In [ ]:
@tool
def compile_clinical_note(
    approved_actions: list,
    safety_verdict: dict,
    monitoring_plan: list,
    uncertainty_disclosures: list,
    critique_feedback: Optional[str] = None,
) -> dict:
    """
    Note builder: assemble the final structured clinical recommendation.
    TERMINAL action on the continue path — call this LAST after safety verification.
    Produces: recommendation summary, actions with safety clearances, monitoring,
    stop conditions, uncertainty disclosures, and follow-up.
    """
    system = """You are a clinical documentation specialist.
Compile a concise, actionable clinical brief from the approved actions and safety verdict.
Return JSON only:
{
  "recommendation_summary": "<1-2 sentence clinical summary>",
  "approved_actions": [{"action": "<action>", "safety_status": "cleared",
                         "monitoring_required": ["<param>"]}],
  "required_monitoring": ["<parameter>"],
  "stop_conditions": ["<condition requiring immediate reassessment>"],
  "uncertainty_disclosures": ["<disclosure>"],
  "follow_up": ["<action>"],
  "clinician_alert": "<urgent note if any, else empty string>"
}"""
    _fb_block = ""
    if critique_feedback:
        _fb_block = (
            "\n\n[CRITIQUE FEEDBACK on the previous note]\n"
            f"{critique_feedback}\n"
            "Fix the issue above SPECIFICALLY in this new note. "
            "Do not reproduce the same problem.")
    user = (
        f"Approved actions: {json.dumps(approved_actions, indent=2)}\n"
        f"Safety verdict: {json.dumps(safety_verdict, indent=2)}\n"
        f"Monitoring plan: {monitoring_plan}\n"
        f"Uncertainty disclosures: {uncertainty_disclosures}"
     + _fb_block
    )
    out = _llm_json(LLM, system, user, {
        "recommendation_summary": "Clinical brief unavailable.",
        "approved_actions": approved_actions,
        "required_monitoring": monitoring_plan,
        "stop_conditions": [], "uncertainty_disclosures": uncertainty_disclosures,
        "follow_up": [], "clinician_alert": ""
    })
    return AgentHandoff(
        tool_name="compile_clinical_note",
        input_received={"approved_actions": approved_actions},
        reasoning_trace="Clinical brief compiled from approved actions and safety clearances.",
        output=out,
        confidence=0.9,
        caveats=uncertainty_disclosures[:3],
        suggested_next_tools=[],
    ).to_dict()

### 6.9 · `activate_emergency`

In [ ]:
@tool
def activate_emergency(reason: str, immediate_actions: list) -> dict:
    """
    Emergency bypass: EXIT the ReAct loop immediately and trigger the emergency alert pathway.
    Call ONLY when assess_case_urgency returns urgency=critical.
    Bypasses Critique Agent and Clinical Brief to preserve <=3s response time.
    Returns: escalation_level, immediate_actions, differential_to_exclude, monitoring_requirements.
    """
    actions = immediate_actions or [
        "Activate rapid response / code team",
        "Establish large-bore IV access x2",
        "Continuous cardiac monitor + pulse oximetry",
        "Physician notification — STAT",
    ]
    payload = {
        "escalation_level": "ICU-immediate",
        "reason": reason,
        "immediate_actions": actions,
        "differential_to_exclude": [
            "Cardiogenic shock", "Tension pneumothorax",
            "Massive PE", "Septic shock", "Haemorrhagic shock",
        ],
        "monitoring_requirements": [
            "HR q1min", "BP q2min", "SpO2 continuous", "Urine output hourly"
        ],
        "emergency_activated": True,
    }
    return AgentHandoff(
        tool_name="activate_emergency",
        input_received={"reason": reason},
        reasoning_trace=f"EMERGENCY ACTIVATED: {reason}",
        output=payload,
        confidence=0.95,
        caveats=["Critique Agent and Clinical Brief bypassed for speed."],
        suggested_next_tools=[],
    ).to_dict()

### 6.10 · Tool registry

In [ ]:
TOOL_LIST = [
    assess_case_urgency, retrieve_clinical_data, draft_treatment_plan,
    screen_drug_conflicts, assess_allergy_risk, render_safety_verdict,
    recall_case_history, compile_clinical_note, activate_emergency,
]
TOOL_MAP  = {t.name: t for t in TOOL_LIST}

print(f"Defined {len(TOOL_LIST)} tools: {[t.name for t in TOOL_LIST]}")

## 7 · Parallel Safety Execution Helper

Runs `screen_drug_conflicts ∥ assess_allergy_risk` in a 2-worker `ThreadPoolExecutor`.
Each side is skipped via a sentinel handoff when the patient has no medications /
no allergies; both skipped → no thread pool.


In [ ]:
_SKIP_D3 = AgentHandoff(
    tool_name="screen_drug_conflicts",
    input_received={}, reasoning_trace="Skipped — no current medications.",
    output={"blocked_actions": [], "caution_actions": [], "risk_factors": [],
            "uncertainty_flags": [], "overall_safety_confidence": 1.0},
    confidence=1.0, caveats=[], suggested_next_tools=["assess_allergy_risk"],
).to_dict()

_SKIP_ALLERGY = AgentHandoff(
    tool_name="assess_allergy_risk",
    input_received={}, reasoning_trace="Skipped — no documented allergies.",
    output={"high_risk_actions": [], "moderate_risk_actions": [],
            "low_risk_actions": [], "safer_alternatives": [],
            "overall_allergy_confidence": 1.0},
    confidence=1.0, caveats=[], suggested_next_tools=["render_safety_verdict"],
).to_dict()


def run_safety_tools_parallel(
    candidate_actions: list,
    current_medications: list,
    documented_allergies: list,
    rag_cases: Optional[list] = None,
) -> tuple[dict, dict]:
    """
    Execute screen_drug_conflicts and assess_allergy_risk concurrently.
    Skips each tool if its precondition is not met (no meds / no allergies).
    Returns (d3_handoff_dict, allergy_handoff_dict).
    """
    has_meds    = bool(current_medications)
    has_allergy = bool(documented_allergies)

    if not has_meds and not has_allergy:
        return _SKIP_D3, _SKIP_ALLERGY

    def _d3():
        if has_meds:
            return screen_drug_conflicts.invoke(
                {"candidate_actions": candidate_actions,
                 "current_medications": current_medications,
                 "rag_cases": rag_cases or []}
            )
        return _SKIP_D3

    def _allergy():
        if has_allergy:
            return assess_allergy_risk.invoke(
                {"candidate_actions": candidate_actions,
                 "documented_allergies": documented_allergies}
            )
        return _SKIP_ALLERGY

    t0 = time.time()
    with ThreadPoolExecutor(max_workers=2) as exe:
        f_d3      = exe.submit(_d3)
        f_allergy = exe.submit(_allergy)
        d3_hd     = f_d3.result()
        ally_hd   = f_allergy.result()
    elapsed = time.time() - t0
    print(f"    [parallel safety] D3({has_meds}) + Allergy({has_allergy}) completed in {elapsed:.2f}s")
    return d3_hd, ally_hd

print("Parallel safety helper defined.")

## 8 · Orchestrator Agent (ReAct Loop)

`LLM` selects tools under 5 mandatory constraints (urgency first → emergency
short-circuit → treatment must pass safety + verdict before the note → low-confidence
fallback → advisory next-tool hints). Bounded at 18 iterations; resumable for one
critique-driven remediation pass.


### 8.1 · Orchestrator system prompt

In [ ]:
ORCHESTRATOR_SYSTEM = """You are the Crisis CDS Orchestrator. Route a patient case through nine tools.

MANDATORY:
1. Call `assess_case_urgency` FIRST. No exceptions.
2. Triage critical AND (conf>=0.75 OR red_flag) -> call `activate_emergency` IMMEDIATELY (exits loop). Otherwise proceed via standard pathway; premature `activate_emergency` is blocked at runtime.
3. `draft_treatment_plan` output MUST pass at least one safety tool (`screen_drug_conflicts` OR `assess_allergy_risk`) BEFORE `compile_clinical_note`.
4. `suggested_next_tools` is ADVISORY — you decide routing.

TOOL ORDER (continue path):
- triage non-critical -> `retrieve_clinical_data` (descriptive clinical query)
- -> `draft_treatment_plan` (symptoms, evidence, meds, allergies)
- patient has meds  -> MUST call `screen_drug_conflicts` before compile_clinical_note
- patient has allergies -> MUST call `assess_allergy_risk` before compile_clinical_note
  (Critique enforces both; skipping triggers a remediation round.)
- both meds+allergies -> call together (they run in parallel)
- -> `render_safety_verdict` (with both safety outputs)
- -> `compile_clinical_note` (terminates loop)

DO NOT CALL:
- `screen_drug_conflicts`: patient has no meds AND no meds in plan
- `assess_allergy_risk`: patient has no allergies
- `recall_case_history`: cold start (no finalised case yet)
- Same tool more than 2x per case
- `retrieve_clinical_data`: call ONCE per case. Internal rewrites already
  exhausted query refinement; re-calling does NOT improve the result.

TERMINATE on `compile_clinical_note` (continue) or `activate_emergency` (escalation).
"""

### 8.2 · ReAct loop

In [ ]:
def _validate_messages(messages: list) -> list:
    """Strip any trailing AIMessage whose tool_calls lack ToolMessage responses.

    Prevents BadRequestError when prior_messages are passed to a remediation run
    and add_messages has deduplicated/dropped a ToolMessage (e.g. parallel pair
    without explicit IDs).
    """
    from langchain_core.messages import AIMessage, ToolMessage as TM
    resolved = {m.tool_call_id for m in messages
                if isinstance(m, TM) and getattr(m, "tool_call_id", None)}
    clean = []
    for msg in messages:
        if isinstance(msg, AIMessage) and getattr(msg, "tool_calls", None):
            missing = [tc["id"] for tc in msg.tool_calls if tc["id"] not in resolved]
            if missing:
                break  # discard this message and everything after it
        clean.append(msg)
    return clean


def orchestrator_react_loop(
    case_state: dict,
    missing_data_flags: list,
    prior_messages: Optional[list] = None,
    prior_state: Optional[dict] = None,
    critique_feedback: Optional[str] = None,
) -> dict:
    """Run the agentic ReAct loop. Returns dict of state updates."""
    # Dynamic tool binding: drop recall_case_history on cold start
    # (memory empty). Its output is consumed only by the router's
    # reasoning; with nothing to recall the call wastes a router
    # iteration.
    _mem_has = bool(getattr(DECISION_MEMORY, "records", None))
    _tools_for_pass = [t for t in TOOL_LIST
                       if t.name != "recall_case_history" or _mem_has]
    llm_with_tools = LLM_THINK.bind_tools(_tools_for_pass)

    if prior_messages and critique_feedback:
        # Remediation retry: resume from existing conversation
        safe_prior = _validate_messages(list(prior_messages))
        messages   = safe_prior + [HumanMessage(content=critique_feedback)]
        msg_offset = len(safe_prior)  # only return messages from here onward
        ps             = prior_state or {}
        triage_result  = ps.get("triage_result")
        rag_result     = ps.get("rag_result")
        treatment_plan = ps.get("treatment_plan")
        d3_result      = ps.get("d3_result")
        allergy_result = ps.get("allergy_result")
        safety_verdict = ps.get("safety_verdict")
        clinical_note  = ps.get("clinical_note")
    else:
        messages = [
            SystemMessage(content=ORCHESTRATOR_SYSTEM),
            HumanMessage(
                content=(
                    f"Patient case:\n{json.dumps(case_state, indent=2)}\n\n"
                    f"Missing data flags: {missing_data_flags}"
                )
            ),
        ]
        msg_offset     = 0
        triage_result  = None
        rag_result     = None
        treatment_plan = None
        d3_result      = None
        allergy_result = None
        safety_verdict = None

    audit_trace:       list[dict]     = []
    clinical_note:     Optional[dict] = None
    emergency_payload: Optional[dict] = None
    is_emergency  = False
    _step         = 0   # cross-iteration tool counter for the live log
    _tool_counts  = {}  # per-tool call counter for the repeat-guard
    _MAX_REPEAT   = 2   # hard runtime cap on calls of any single tool
    _nudged       = False  # cap at one premature-end nudge per case
    terminated    = False
    iteration     = 0
    MAX_ITER      = 18
    rag_tool_msg  = None       # full RAG notes sent once, then
    rag_age       = 0          # compacted (not resent every step)
    _rag_compacted = False

    while not terminated and iteration < MAX_ITER:
        iteration += 1

        if rag_tool_msg is not None and rag_age >= 1 and not _rag_compacted:
            # Full retrieved notes were in context for the post-
            # retrieval decision (incl. draft); downstream tools read
            # them from state. Compact so the uncapped notes are not
            # re-sent on every remaining ReAct iteration.
            _f = json.loads(rag_tool_msg.content)
            rag_tool_msg.content = json.dumps({
                "note": "retrieved evidence already delivered above "
                        "and available to downstream tools",
                "relevance_score": _f.get("relevance_score"),
                "cases": [{"id": c.get("id"),
                           "similarity": c.get("similarity")}
                          for c in _f.get("cases", [])],
            })
            _rag_compacted = True
        response = llm_with_tools.invoke(messages)
        if rag_tool_msg is not None:
            rag_age += 1
        messages.append(response)

        print(f"\n  ▼ ORCHESTRATOR · iteration {iteration}")
        if not response.tool_calls:
            # If the loop is ending PREMATURELY on the continue path,
            # nudge the LLM with an explicit instruction once.
            _tools_done = {h.get("tool_name") for h in audit_trace}
            _missing = None
            if (not is_emergency and not _nudged
                    and treatment_plan is not None
                    and "compile_clinical_note" not in _tools_done):
                _has_meds  = bool((case_state or {}).get("medications"))
                _has_allgs = bool((case_state or {}).get("allergies"))
                if _has_meds and "screen_drug_conflicts" not in _tools_done:
                    _missing = "screen_drug_conflicts"
                elif _has_allgs and "assess_allergy_risk" not in _tools_done:
                    _missing = "assess_allergy_risk"
                elif "render_safety_verdict" not in _tools_done:
                    _missing = "render_safety_verdict"
                else:
                    _missing = "compile_clinical_note"
            if _missing:
                print(f"    · loop wants to end but {_missing!r} is missing — nudging once")
                _nudge_text = (
                    "You have not called `" + _missing + "` yet. "
                    "The pipeline requires it before terminating. "
                    "Call `" + _missing + "` now."
                )
                messages.append(HumanMessage(content=_nudge_text))
                _nudged = True
                continue   # retry the iteration with the nudge in context
            print("    · loop ends (no tool calls)")
            break

        # Hard runtime repeat-guard: if any tool has been requested more
        # than _MAX_REPEAT times across the case, terminate. Same-tool
        # retries on the SAME inputs never produce a different result;
        # the LLM router otherwise loops to MAX_ITER on low-confidence tools.
        _over = []
        for _tc in response.tool_calls:
            _n = _tc["name"]
            _tool_counts[_n] = _tool_counts.get(_n, 0) + 1
            if _tool_counts[_n] > _MAX_REPEAT:
                _over.append(_n)
        if _over:
            print(f"    · repeat-guard tripped on {sorted(set(_over))} "
                  f"(>{_MAX_REPEAT} calls) — terminating loop")
            terminated = True
            for _tc in response.tool_calls:
                messages.append(ToolMessage(
                    content='{"status": "cancelled", "reason": "repeat-guard"}',
                    tool_call_id=_tc["id"], id=str(uuid.uuid4())))
            break

        call_names = [tc["name"] for tc in response.tool_calls]

        # Parallel safety pair detection
        safety_pair = {"screen_drug_conflicts", "assess_allergy_risk"}
        if safety_pair.issubset(set(call_names)) and len(response.tool_calls) == 2:
            d3_tc   = next(tc for tc in response.tool_calls if tc["name"] == "screen_drug_conflicts")
            ally_tc = next(tc for tc in response.tool_calls if tc["name"] == "assess_allergy_risk")

            # candidate_actions fallback to drafted plan (mirrors the
            # sequential elif arg-injection — see audit: parallel branch
            # was bypassing all arg-injection fixes).
            acts  = (d3_tc["args"].get("candidate_actions")
                     or ally_tc["args"].get("candidate_actions")
                     or ((treatment_plan or {}).get("output", {})
                          or {}).get("candidate_actions", []))
            meds  = d3_tc["args"].get("current_medications",
                          case_state.get("medications", []))
            allgs = ally_tc["args"].get("documented_allergies",
                          case_state.get("allergies", []))

            rag_cases_for_d3 = (rag_result or {}).get("output", {}).get("cases", [])
            d3_hd, ally_hd = run_safety_tools_parallel(
                acts, meds, allgs, rag_cases=rag_cases_for_d3
            )
            d3_result       = d3_hd
            allergy_result  = ally_hd
            audit_trace.extend([d3_hd, ally_hd])

            messages.append(ToolMessage(content=json.dumps(d3_hd["output"]),
                                        tool_call_id=d3_tc["id"],
                                        id=str(uuid.uuid4())))
            messages.append(ToolMessage(content=json.dumps(ally_hd["output"]),
                                        tool_call_id=ally_tc["id"],
                                        id=str(uuid.uuid4())))
            continue

        # Sequential tool execution
        for idx_tc, tc in enumerate(response.tool_calls):
            name = tc["name"]
            args = tc["args"]

            if name not in TOOL_MAP:
                messages.append(ToolMessage(
                    content=f"Error: unknown tool '{name}'.",
                    tool_call_id=tc["id"],
                    id=str(uuid.uuid4())))
                continue

            # Emergency short-circuit — GATED. Honour activate_emergency
            # only when triage was high-confidence critical (ESI-1) or
            # flagged an objective resuscitation red flag. A low-confidence
            # or non-critical bypass is blocked and redirected to the
            # standard pathway so the critique agent still reviews.
            if name == "activate_emergency":
                _tri      = (triage_result or {}).get("output", {}) or {}
                _tri_conf = float((triage_result or {}).get("confidence", 0.0) or 0.0)
                _allow_em = (_tri.get("urgency") == "critical"
                             and (_tri_conf >= 0.75 or bool(_tri.get("red_flags"))))
                if not _allow_em:
                    print(f"  ·  activate_emergency BLOCKED "
                          f"(urgency={_tri.get('urgency')!r} conf={_tri_conf:.2f}, no red_flag)")
                    messages.append(ToolMessage(
                        content=json.dumps({
                            "status": "blocked",
                            "reason": ("Emergency bypass requires high-confidence "
                                       "critical triage (urgency=critical, "
                                       "confidence>=0.75) or an objective "
                                       "resuscitation red flag. This case did "
                                       "not meet that bar."),
                            "triage_urgency": _tri.get("urgency"),
                            "triage_confidence": _tri_conf,
                            "next": ("Proceed with the standard pathway: "
                                     "retrieve_clinical_data -> "
                                     "draft_treatment_plan -> safety tools -> "
                                     "render_safety_verdict -> "
                                     "compile_clinical_note."),
                        }),
                        tool_call_id=tc["id"],
                        id=str(uuid.uuid4())))
                    continue
                hd = activate_emergency.invoke(args)
                audit_trace.append(hd)
                emergency_payload = hd["output"]
                is_emergency = True
                terminated   = True
                messages.append(ToolMessage(content=json.dumps(hd["output"]),
                                            tool_call_id=tc["id"]))
                for _rtc in response.tool_calls[idx_tc + 1:]:
                    messages.append(ToolMessage(
                        content='{"status": "cancelled", "reason": "emergency activated"}',
                        tool_call_id=_rtc["id"],
                        id=str(uuid.uuid4())))
                print(f"  ·  EMERGENCY ACTIVATED — loop exits.")
                break

            # Inject accumulated state so LLM doesn't need to re-pass prior outputs
            _pre_handoff = None  # set by precondition-skip branches below
            if name == "assess_case_urgency":
                # All triage inputs are in case_state. Inject from there so the
                # router cannot silently drop acuity_signals — the rubric needs
                # them as objective escalation cues when vitals are absent.
                args = dict(args)
                args.setdefault("symptoms",       case_state.get("symptoms", []))
                args.setdefault("vitals",         case_state.get("vitals", {}))
                args.setdefault("medications",    case_state.get("medications", []))
                args.setdefault("allergies",      case_state.get("allergies", []))
                args.setdefault("missing_data",   missing_data_flags or [])
                args.setdefault("acuity_signals", case_state.get("acuity_signals", []))

            elif name == "render_safety_verdict":
                args = dict(args)
                # urgency + candidate_actions are REQUIRED by the tool. Inject
                # from triage_result and treatment_plan so the router does not
                # waste iterations on pydantic ValidationErrors.
                args.setdefault("urgency",
                    ((triage_result or {}).get("output", {}) or {}).get("urgency", "urgent"))
                args.setdefault("candidate_actions",
                    ((treatment_plan or {}).get("output", {}) or {}).get("candidate_actions", []))
                args.setdefault("d3_output",     (d3_result      or {}).get("output", {}))
                args.setdefault("allergy_output", (allergy_result or {}).get("output", {}))

            elif name == "compile_clinical_note":
                args = dict(args)
                sv_out = (safety_verdict or {}).get("output", {})
                # approved_actions is REQUIRED. Pull from the safety verdict
                # output (preferred) or fall back to the drafted candidates.
                args.setdefault("approved_actions",
                    sv_out.get("approved_actions")
                    or ((treatment_plan or {}).get("output", {}) or {}).get("candidate_actions", []))
                args.setdefault("safety_verdict",          sv_out)
                args.setdefault("monitoring_plan",         sv_out.get("required_monitoring", []))
                args.setdefault("uncertainty_disclosures", sv_out.get("stop_conditions", []))
                if critique_feedback:
                    args.setdefault("critique_feedback", critique_feedback)

            elif name == "screen_drug_conflicts":
                args = dict(args)
                args.setdefault(
                    "rag_cases",
                    (rag_result or {}).get("output", {}).get("cases", [])
                )
                args.setdefault("current_medications",
                                case_state.get("medications", []))
                args.setdefault("candidate_actions",
                                (treatment_plan or {}).get("output", {})
                                .get("candidate_actions", []))
                # Precondition skip: if the patient has no documented meds,
                # do NOT run the live tool — RAG-extracted drugs are not the
                # patient's and the tool can emit placeholder "BLOCKED"
                # entries. Substitute the _SKIP_D3 sentinel handoff.
                if not args.get("current_medications"):
                    _pre_handoff = _SKIP_D3

            elif name == "assess_allergy_risk":
                args = dict(args)
                args.setdefault("documented_allergies",
                                case_state.get("allergies", []))
                args.setdefault("candidate_actions",
                                (treatment_plan or {}).get("output", {})
                                .get("candidate_actions", []))
                # Precondition skip: if the patient has no documented
                # allergies, the live grader has nothing to evaluate and
                # returns a vacuous "0.90 confidence" with empty risk lists.
                # Use the _SKIP_ALLERGY sentinel instead.
                if not args.get("documented_allergies"):
                    _pre_handoff = _SKIP_ALLERGY

            elif name == "draft_treatment_plan":
                args = dict(args)
                args.setdefault("symptoms", case_state.get("symptoms", []))
                args.setdefault("current_medications",
                                case_state.get("medications", []))
                args.setdefault("documented_allergies",
                                case_state.get("allergies", []))
                _rag_out = (rag_result or {}).get("output", {}) or {}
                args.setdefault("retrieved_evidence", json.dumps(_rag_out.get("cases", [])))
                args.setdefault("rag_drug_patterns", _rag_out.get("drug_patterns", []))
                if critique_feedback:
                    args.setdefault("critique_feedback", critique_feedback)

            _t_tool = time.time()
            try:
                hd = _pre_handoff if _pre_handoff is not None else TOOL_MAP[name].invoke(args)
            except Exception as _tool_err:
                _msg = f"{type(_tool_err).__name__}: {_tool_err}"
                print(f"  ·  {name} retry (arg error: {_msg[:80]})")
                messages.append(ToolMessage(
                    content=json.dumps({
                        "error": _msg,
                        "hint": "Tool call failed (often a missing/invalid "
                                "argument). Correct the arguments and retry, "
                                "or proceed with a different tool."}),
                    tool_call_id=tc["id"], id=str(uuid.uuid4())))
                continue
            audit_trace.append(hd)
            _step += 1
            _cf = hd.get("confidence", 0.0) or 0.0
            _st = "HIGH" if _cf >= 0.75 else "OK  " if _cf >= 0.55 else "LOW " if _cf >= 0.30 else "----"
            _xt = ""
            if name == "retrieve_clinical_data":
                _o = hd.get("output", {}) or {}
                _xt = f"   {len(_o.get('cases', []))} cases · score {_o.get('relevance_score', 0):.2f}"
                if _o.get("rewrites"):
                    _xt += f" · rwx{len(_o['rewrites'])}"
            elif name == "screen_drug_conflicts":
                _f = (hd.get("output", {}) or {}).get("findings", [])
                if _f:
                    _xt = f"   {len(_f)} DDI"
            elif name == "assess_case_urgency":
                _u = (hd.get("output", {}) or {}).get("urgency", "?")
                _xt = f"   → {_u}"
            _dur = time.time() - _t_tool
            _xt_clean = _xt.strip()
            _tail = ("  · " + _xt_clean) if _xt_clean else ""
            print(f"    └─ ▶ TOOL  {name:<28} {_dur:>5.1f}s  conf {_cf:>4.2f} [{_st.strip()}]{_tail}")

            if name == "assess_case_urgency":     triage_result  = hd
            elif name == "retrieve_clinical_data": rag_result     = hd
            elif name == "draft_treatment_plan":   treatment_plan = hd
            elif name == "screen_drug_conflicts":  d3_result      = hd
            elif name == "assess_allergy_risk":    allergy_result = hd
            elif name == "render_safety_verdict":  safety_verdict = hd
            elif name == "compile_clinical_note":
                clinical_note = hd
                terminated    = True

            messages.append(ToolMessage(content=json.dumps(hd["output"]),
                                        tool_call_id=tc["id"],
                                        id=str(uuid.uuid4())))
            if name == "retrieve_clinical_data":
                rag_tool_msg, rag_age, _rag_compacted = messages[-1], 0, False
            if terminated:
                for _rtc in response.tool_calls[idx_tc + 1:]:
                    messages.append(ToolMessage(
                        content='{"status": "cancelled", "reason": "loop terminated"}',
                        tool_call_id=_rtc["id"],
                        id=str(uuid.uuid4())))
                break

    return {
        "messages":          messages[msg_offset:],
        "audit_trace":       audit_trace,
        "triage_result":     triage_result,
        "rag_result":        rag_result,
        "treatment_plan":    treatment_plan,
        "d3_result":         d3_result,
        "allergy_result":    allergy_result,
        "safety_verdict":    safety_verdict,
        "clinical_note":     clinical_note,
        "emergency_payload": emergency_payload,
        "is_emergency":      is_emergency,
    }

print("Orchestrator ReAct loop defined.")

## 9 · Critique Agent (post-loop)

Five checks over the audit trace: safety-path completeness, no blocked action in the
note, RAG used for non-critical cases, medical plausibility (HF reviewer model), and
unverifiable medications/allergens → forces a pharmacist-review flag. Failure triggers
one orchestrator remediation pass; bypassed entirely on the emergency path.


### 9.1 · HuggingFace medical critique (optional)

In [ ]:
# HuggingFace medical model client
import os, json as _json, re as _re

_HF_MEDICAL_MODEL = os.getenv("HF_MEDICAL_MODEL", "Qwen/Qwen2.5-72B-Instruct")
_HF_PROVIDER       = os.getenv("HF_PROVIDER",       "novita")
_HF_TOKEN         = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_API_TOKEN") or ""

try:
    from huggingface_hub import InferenceClient as _HFClient
    _HF_CLIENT = _HFClient(model=_HF_MEDICAL_MODEL, provider=_HF_PROVIDER, token=_HF_TOKEN or None)
except Exception:
    _HF_CLIENT = None  # HF SDK not installed


def _run_hf_medical_critique(medications, candidate_actions, approved_actions, symptoms, urgency):
    if _HF_CLIENT is None:
        return {"suspicious_drugs": [], "clinical_issues": [], "passed": True,
                "reasoning": "HF client unavailable"}
    def _s(x): return x.get("action", x.get("name", str(x))) if isinstance(x, dict) else str(x)
    med_list    = ", ".join(_s(m) for m in medications)       if medications       else "none"
    action_list = ", ".join(_s(a) for a in candidate_actions) if candidate_actions else "none"
    approved    = ", ".join(_s(a) for a in approved_actions)  if approved_actions  else "none"
    sym_list    = ", ".join(_s(s) for s in symptoms)          if symptoms          else "unspecified"
    system_msg = ("You are a clinical pharmacologist and safety reviewer. "
                  "Reply ONLY with a JSON object, no markdown, no prose.")
    user_msg = (
        f"Review clinical data for a {urgency or 'unknown'} urgency case.\n\n"
        f"Symptoms: {sym_list}\nMedications: {med_list}\n"
        f"Candidate actions: {action_list}\nApproved: {approved}\n\n"
        "Identify: 1) fabricated/misspelled drug names. 2) implausible clinical actions.\n"
        'Respond with JSON only: {"suspicious_drugs":[],"clinical_issues":[],"passed":true,"reasoning":""}\n'
        "Set passed=false if either list is non-empty."
    )
    try:
        resp = _HF_CLIENT.chat_completion(
            messages=[{"role": "system", "content": system_msg},
                      {"role": "user",   "content": user_msg}],
            max_tokens=400, temperature=0.0)
        raw = resp.choices[0].message.content
    except Exception as e:
        return {"suspicious_drugs": [], "clinical_issues": [], "passed": True,
                "reasoning": f"HF API error: {e}"}
    raw = _re.sub(r"```(?:json)?", "", raw).strip()
    try:
        data = _json.loads(raw)
        return {"suspicious_drugs": data.get("suspicious_drugs", []),
                "clinical_issues":  data.get("clinical_issues",  []),
                "passed":           bool(data.get("passed", True)),
                "reasoning":        str(data.get("reasoning", ""))}
    except Exception:
        return {"suspicious_drugs": [], "clinical_issues": [], "passed": True,
                "reasoning": f"JSON parse error: {raw[:120]}"}

### 9.2 · Critique agent — 5 checks

In [ ]:
def run_critique_agent(audit_trace, clinical_note, case_state=None):
    tools_called = [h["tool_name"] for h in audit_trace]

    def _is_skipped(name):
        for h in audit_trace:
            if h["tool_name"] == name:
                return "Skipped" in h.get("reasoning_trace", "")
        return True

    n_complete   = sum(1 for h in audit_trace
                       if h.get("reasoning_trace") and h.get("output") is not None
                       and h.get("confidence") is not None)
    completeness = n_complete / max(len(audit_trace), 1)
    checks: list = []
    remediation_steps: list = []

    # Check 1: safety_path_complete
    if "draft_treatment_plan" in tools_called:
        has_meds  = bool((case_state or {}).get("medications"))
        has_allgs = bool((case_state or {}).get("allergies"))
        d3_ok   = "screen_drug_conflicts" in tools_called
        ally_ok = "assess_allergy_risk"   in tools_called
        sv_ok   = "render_safety_verdict" in tools_called
        # Require d3 only when patient has meds; ally only when has allergies.
        # Mirrors the orchestrator prompt: "if meds → screen, if allergies → assess".
        d3_need_ok   = d3_ok   or not has_meds
        ally_need_ok = ally_ok or not has_allgs
        if d3_need_ok and ally_need_ok and sv_ok:
            checks.append({"name": "safety_path_complete", "passed": True, "issue": ""})
        else:
            missing = []
            if has_meds  and not d3_ok:   missing.append("screen_drug_conflicts")
            if has_allgs and not ally_ok: missing.append("assess_allergy_risk")
            if not sv_ok:                 missing.append("render_safety_verdict")
            missing.append("compile_clinical_note")
            issue = f"Treatment plan not safety-screened; call: {missing}"
            checks.append({"name": "safety_path_complete", "passed": False, "issue": issue})
            for t in missing: remediation_steps.append({"tool": t, "reason": issue})
    else:
        checks.append({"name": "safety_path_complete", "passed": True,
                       "issue": "draft_treatment_plan absent"})

    # Check 2: no_blocked_action_in_note
    d3_ran   = "screen_drug_conflicts" in tools_called and not _is_skipped("screen_drug_conflicts")
    note_ran = "compile_clinical_note" in tools_called
    if d3_ran and note_ran:
        blocked_raw = []
        for h in audit_trace:
            if h["tool_name"] == "screen_drug_conflicts":
                blocked_raw = h.get("output", {}).get("blocked_actions", [])
                break
        blocked_word_sets = [
            set(str(b.get("action", b) if isinstance(b, dict) else b).lower().split()[:4])
            for b in blocked_raw]
        cn_out   = ((clinical_note or {}).get("output") or {})
        approved = [str(a.get("action", a) if isinstance(a, dict) else a).lower()
                    for a in cn_out.get("approved_actions", [])]
        leaked = []
        for bwords, braw in zip(blocked_word_sets, blocked_raw):
            for appr in approved:
                if bwords & set(appr.split()):
                    leaked.append(str(braw.get("action", braw) if isinstance(braw, dict) else braw))
                    break
        if leaked:
            issue = f"Blocked action(s) in compiled note: {leaked[:3]}"
            checks.append({"name": "no_blocked_in_note", "passed": False, "issue": issue})
            remediation_steps.append({"tool": "compile_clinical_note",
                                       "reason": issue + " re-compile excluding blocked"})
        else:
            checks.append({"name": "no_blocked_in_note", "passed": True, "issue": ""})
    else:
        checks.append({"name": "no_blocked_in_note", "passed": True, "issue": "not applicable"})

    # Check 3: rag_called_if_noncritical
    triage_urgency = ""
    for h in audit_trace:
        if h["tool_name"] == "assess_case_urgency":
            triage_urgency = (h.get("output") or {}).get("urgency", "")
            break
    if triage_urgency and triage_urgency.lower() != "critical":
        rag_ran = ("retrieve_clinical_data" in tools_called
                   and not _is_skipped("retrieve_clinical_data"))
        if rag_ran:
            checks.append({"name": "rag_called_if_noncritical", "passed": True, "issue": ""})
        else:
            issue = f"Non-critical ({triage_urgency}) skipped evidence retrieval"
            checks.append({"name": "rag_called_if_noncritical", "passed": False, "issue": issue})
            remediation_steps.insert(0, {"tool": "retrieve_clinical_data", "reason": issue})
    else:
        checks.append({"name": "rag_called_if_noncritical", "passed": True,
                       "issue": "Critical or unknown urgency"})

    # Check 4: medical_plausibility (HuggingFace OpenBioLLM)
    meds_found: list = []
    cands_found: list = []
    appr_found: list = []
    syms_found: list = []
    for h in audit_trace:
        out = h.get("output") or {}
        if h["tool_name"] == "draft_treatment_plan":
            meds_found  = out.get("medications",       [])
            cands_found = out.get("candidate_actions", [])
        elif h["tool_name"] == "compile_clinical_note":
            appr_found  = out.get("approved_actions",  [])
        elif h["tool_name"] == "assess_case_urgency":
            syms_found  = out.get("symptoms",          [])
    if meds_found or cands_found:
        hf = _run_hf_medical_critique(
            medications=meds_found, candidate_actions=cands_found,
            approved_actions=appr_found, symptoms=syms_found, urgency=triage_urgency)
        if hf["passed"]:
            checks.append({"name": "medical_plausibility", "passed": True,
                           "issue": (hf["reasoning"] or "")[:80]})
        else:
            bad_d = hf.get("suspicious_drugs", [])
            bad_a = hf.get("clinical_issues",  [])
            parts = []
            if bad_d: parts.append(f"suspicious drugs: {bad_d}")
            if bad_a: parts.append(f"clinical issues: {bad_a[:2]}")
            issue = "; ".join(parts) or hf["reasoning"]
            checks.append({"name": "medical_plausibility", "passed": False, "issue": issue})
            remediation_steps.append({"tool": "draft_treatment_plan",
                "reason": f"Medical plausibility failed: {issue}. Use real medications."})
    else:
        checks.append({"name": "medical_plausibility", "passed": True,
                       "issue": "no medications/actions found"})


    # Check 5: unverifiable_medications
    # Fires when screen_drug_conflicts cannot verify a drug's identity in any
    # clinical database (RxNav lookup failed, pharmacologic class unknown) OR
    # when assess_allergy_risk returns near-zero confidence on a listed allergen.
    # Neither case can be auto-remediated by the orchestrator; the critique
    # forces a PHARMACIST REVIEW REQUIRED flag into the compiled note.
    _UNVERIF_RE = _re.compile(
        r"not retrieved|identity[/\s]class|pharmacologic\s+identity|"
        r"no information on actual\s+pharmacologic|rxnav\s+ddi|"
        r"not in.*database|unrecognized|identity of|"
        r"exact active ingredient|"
        r"pharmacologic class\w*\s+(?:for\s+\S+\s+)?not\s+(?:provided|known|available|confirmed)|"
        r"active ingredient\w*\s+(?:for\s+\S+\s+)?not\s+(?:provided|known|available|confirmed)",
        _re.IGNORECASE,
    )
    d3_entry    = next((h for h in audit_trace if h["tool_name"] == "screen_drug_conflicts"), None)
    ally_entry  = next((h for h in audit_trace if h["tool_name"] == "assess_allergy_risk"),   None)
    cs          = case_state or {}
    has_meds    = bool(cs.get("medications", []))
    has_allergy = bool(cs.get("allergies",   []))

    unverifiable_drugs     = []
    unverifiable_allergens = []

    if d3_entry and has_meds:
        d3_conf    = d3_entry.get("confidence", 1.0)
        d3_caveats = " ".join(d3_entry.get("caveats", []))
        if d3_conf < 0.50 and _UNVERIF_RE.search(d3_caveats):
            for m in cs.get("medications", []):
                raw_name  = m.get("name", str(m)) if isinstance(m, dict) else str(m)
                drug_name = raw_name.split()[0]
                if _re.search(rf"\b{_re.escape(drug_name)}\b", d3_caveats, _re.IGNORECASE):
                    unverifiable_drugs.append(raw_name)

    if ally_entry and has_allergy:
        ally_conf = ally_entry.get("confidence", 1.0)
        if ally_conf < 0.30:
            for a in cs.get("allergies", []):
                allergen = a.get("allergen", str(a)) if isinstance(a, dict) else str(a)
                unverifiable_allergens.append(allergen)

    if unverifiable_drugs or unverifiable_allergens:
        parts = []
        if unverifiable_drugs:
            parts.append(f"medications not in clinical database: {unverifiable_drugs}")
        if unverifiable_allergens:
            parts.append(f"allergens unverifiable (conf<0.30): {unverifiable_allergens}")
        issue = "; ".join(parts)
        checks.append({"name": "unverifiable_medications", "passed": False, "issue": issue})
        remediation_steps.append({
            "tool":   "compile_clinical_note",
            "reason": (
                f"{issue}. "
                "Recompile the clinical note and prepend the following mandatory flag: "
                "'\u26a0\ufe0f PHARMACIST REVIEW REQUIRED — one or more medication or "
                "allergen identities could not be verified in any clinical database. "
                "All drug-specific recommendations must be reviewed and confirmed by a "
                "pharmacist before clinical action is taken.' "
                "Remove or clearly caveat any drug-specific dosing, interaction, or "
                "allergy-clearance claims."
            ),
        })
    else:
        checks.append({"name": "unverifiable_medications", "passed": True, "issue": ""})

    passed  = all(c["passed"] for c in checks)
    failed  = [c for c in checks if not c["passed"]]
    summary = (f"All {len(checks)} checks passed." if passed else
               f"{len(failed)}/{len(checks)} failed. " + "; ".join(c["issue"] for c in failed))
    return {
        "passed":             passed,
        "checks":             checks,
        "remediation_steps":  remediation_steps,
        "audit_completeness": completeness,
        "recommendation":     "finalise" if passed else "remediate",
        "critique_summary":   summary,
        "bypass":             False,
    }

print("Critique agent (5 checks) loaded.")

## 10 · Decision Memory (FAISS, in-session)

Ephemeral `IndexFlatIP` store of finalised briefs keyed by a case-summary embedding;
queried by `recall_case_history`. Resets each process — not persisted.


In [ ]:
import threading as _threading

class DecisionMemory:
    """FAISS-backed in-memory store for past clinical decisions (thread-safe)."""

    _lock = _threading.Lock()

    def __init__(self, dim: int = EMBED_DIM):
        self.index   = faiss.IndexFlatIP(dim)
        self.records: list[dict] = []
        self.dim     = dim

    def write(self, case_state: dict, clinical_brief: dict) -> int:
        """Store a finalised Clinical Brief keyed by case embedding. Returns record index."""
        summary = self._summarise(case_state)
        vec     = embed(summary).reshape(1, -1)
        rec = {
            "summary":        summary,
            "brief_summary":  clinical_brief.get("output", clinical_brief)
                              .get("recommendation_summary", "")[:300],
            "verdict":        clinical_brief.get("output", clinical_brief).get("verdict", "unknown"),
            "outcome":        None,
            "timestamp":      datetime.now().isoformat(),
        }
        with DecisionMemory._lock:
            self.index.add(vec)
            self.records.append(rec)
            return len(self.records) - 1

    def query(self, case_summary: str, k: int = 3) -> list[dict]:
        """Return k most similar past decisions."""
        if self.index.ntotal == 0:
            return []
        vec = embed(case_summary).reshape(1, -1)
        with DecisionMemory._lock:
            if self.index.ntotal == 0:
                return []
            k_           = min(k, self.index.ntotal)
            scores, idxs = self.index.search(vec, k_)
            hits = [(float(sc), self.records[i])
                    for sc, i in zip(scores[0], idxs[0]) if i >= 0]
        return [{**r, "similarity": sc} for sc, r in hits]

    def record_outcome(self, idx: int, outcome: str) -> None:
        if 0 <= idx < len(self.records):
            self.records[idx]["outcome"] = outcome

    @staticmethod
    def _summarise(cs: dict) -> str:
        syms   = ", ".join(cs.get("symptoms", []))
        vitals = json.dumps(cs.get("vitals", {}))
        meds   = ", ".join(
            m.get("name", "") if isinstance(m, dict) else str(m)
            for m in cs.get("medications", [])
        )
        allgs  = ", ".join(
            a.get("allergen", "") if isinstance(a, dict) else str(a)
            for a in cs.get("allergies", [])
        )
        return f"Symptoms: {syms}. Vitals: {vitals}. Meds: {meds}. Allergies: {allgs}."


DECISION_MEMORY = DecisionMemory()
print(f"Decision Memory initialised (FAISS IndexFlatIP, dim={EMBED_DIM}).")

## 11 · LangGraph Graph Assembly

Thin outer graph; the real logic is the orchestrator's ReAct loop.
```
START → intake → orchestrator ─┬─ emergency → emergency_alert ─┐
                               ├─ skip_critique ───────────────┤
                               └─ critique ─┬─ remediate (×1) → orchestrator
                                             └─ pass ──────────┴→ finalize → END
```


### 11.1 · Graph nodes & routers

In [ ]:
# Node definitions

def intake_node(state: CrisisOrchestratorState) -> dict:
    """LLM-based intake parser (free text) or direct pass-through (dict) — runs outside the ReAct loop."""
    raw          = state.get("case_state", {})
    cs, missing  = parse_intake(raw)
    thread_id    = state.get("thread_id", str(uuid.uuid4()))

    meds   = [m.get("name", "") if isinstance(m, dict) else str(m) for m in cs.get("medications", [])]
    allgs  = [a.get("allergen", "") if isinstance(a, dict) else str(a) for a in cs.get("allergies", [])]
    syms = cs.get('symptoms', [])
    # intake details printed by run_trace; node stays silent
    return {"case_state": cs, "missing_data_flags": missing, "thread_id": thread_id}

def orchestrator_node(state: CrisisOrchestratorState) -> dict:
    """Run the full ReAct loop over 9 tools. Handles initial run and critique-driven remediation."""
    critique_feedback = state.get("critique_feedback")
    if critique_feedback:
        print("\n  ▼ ORCHESTRATOR (remediation)")
        print("    ├─ feedback: critique flagged remediation steps")
        prior_messages = list(state.get("messages", []))
        prior_state = {
            "triage_result":  state.get("triage_result"),
            "rag_result":     state.get("rag_result"),
            "treatment_plan": state.get("treatment_plan"),
            "d3_result":      state.get("d3_result"),
            "allergy_result": state.get("allergy_result"),
            "safety_verdict": state.get("safety_verdict"),
            "clinical_note":  state.get("clinical_note"),
        }
        result = orchestrator_react_loop(
            state["case_state"],
            state.get("missing_data_flags", []),
            prior_messages=prior_messages,
            prior_state=prior_state,
            critique_feedback=critique_feedback,
        )
        result["critique_feedback"] = None  # clear so route_after_critique does not loop again
        return result
    pass  # ReAct loop starts (no banner)
    return orchestrator_react_loop(
        state["case_state"],
        state.get("missing_data_flags", [])
    )


def critique_node(state: CrisisOrchestratorState) -> dict:
    """Post-loop Critique Agent — skipped on emergency path."""
    if state.get("is_emergency"):
        print(" · critique bypassed (emergency path)")
        return {"critique_report": {"bypass": True, "passed": True,
                                    "reason": "emergency_path", "audit_completeness": 1.0}}
    retry = state.get("critique_retry_count", 0)
    print(f"\n  ─── CRITIQUE ── attempt {retry + 1}/2 " + "─" * 40)
    report = run_critique_agent(
        state.get("audit_trace", []),
        state.get("clinical_note"),
        state.get("case_state"),
    )
    status = "PASS" if report["passed"] else "FAIL"
    print(f"  [{status}]  completeness={report['audit_completeness']:.2%}")
    for c in report["checks"]:
        mark  = "ok  " if c["passed"] else "FAIL"
        label = c.get("issue") or "passed"
        print(f"    [{mark}] {c['name']}: {label}")
    updates: dict = {"critique_report": report}
    if not report["passed"] and retry < 1:
        tools_done = [h["tool_name"] for h in state.get("audit_trace", [])]
        steps      = report["remediation_steps"]
        feedback   = (
            "CRITIQUE AGENT FEEDBACK — remediation required before finalising.\n\n"
            f"Tools already executed (do NOT repeat): {tools_done}\n\n"
            "Failed checks:\n"
            + "".join(f"  - {c['name']}: {c['issue']}\n"
                      for c in report["checks"] if not c["passed"])
            + "\nRequired remediation — call ONLY these tools, in this order:\n"
            + "".join(f"  {i+1}. {s['tool']}: {s['reason']}\n"
                      for i, s in enumerate(steps))
            + "\nDo NOT call assess_case_urgency or any already-completed tool. "
              "Call the listed tools then stop."
        )
        updates["critique_feedback"]    = feedback
        updates["critique_retry_count"] = retry + 1
        _tools = [s["tool"] for s in steps]
        print(f"    → FAIL  ·  routing back to orchestrator · steps: {_tools}")
    return updates


def emergency_alert_node(state: CrisisOrchestratorState) -> dict:
    """Log emergency trigger ? full output in CLINICAL RECOMMENDATION section."""
    p     = state.get("emergency_payload", {})
    level = p.get("escalation_level", "ICU-immediate")
    n_act = len(p.get("immediate_actions", []))
    print(f"   EMERGENCY · level={level} · {n_act} actions · critique+brief bypassed")
    return {}


def finalize_node(state: CrisisOrchestratorState) -> dict:
    """Write Clinical Brief to Decision Memory and print decision path."""
    # If unverifiable_medications critique check failed, stamp the clinician_alert
    # field directly — the LLM may not reliably populate it from feedback alone.
    cr = state.get("critique_report") or {}
    if not cr.get("bypass"):
        for chk in cr.get("checks", []):
            if chk.get("name") == "unverifiable_medications" and not chk.get("passed"):
                cn = state.get("clinical_note") or {}
                cn_out = cn.get("output", cn)
                flag = ("⚠️ PHARMACIST REVIEW REQUIRED — "
                        "one or more medication or allergen identities "
                        "could not be verified in any clinical database. "
                        "Do not administer until identities are confirmed.")
                existing = cn_out.get("clinician_alert", "") or ""
                if flag not in existing:
                    cn_out["clinician_alert"] = flag
                break
    # _triage_unparsed → clinician_alert (parser fell back to default "urgent")
    tri_out = (state.get("triage_result") or {}).get("output", {}) or {}
    if tri_out.get("_triage_unparsed"):
        cn = state.get("clinical_note") or {}
        cn_out = cn.get("output", cn)
        flag = ("⚠️ TRIAGE PARSER FAILED — urgency tier defaulted to "
                "'urgent' at low confidence. Please reassess manually.")
        existing = cn_out.get("clinician_alert", "") or ""
        if flag not in existing:
            cn_out["clinician_alert"] = (existing + " " + flag).strip()

    # Memory-write gating: only record decisions the critique actually passed,
    # OR genuine emergency-path activations. Failed critiques would pollute
    # recall_case_history with bad decisions for future cases.
    crep = state.get("critique_report") or {}
    write_ok = bool(state.get("is_emergency")) or bool(crep.get("passed"))
    brief = state.get("clinical_note") or state.get("emergency_payload") or {}
    if write_ok:
        idx = DECISION_MEMORY.write(state["case_state"], brief)
    else:
        idx = -1   # skipped: critique did not pass

    audit       = state.get("audit_trace", [])
    tools_used  = [h["tool_name"] for h in audit]
    d3_status   = "executed" if "screen_drug_conflicts" in tools_used else "skipped"
    ally_status = "executed" if "assess_allergy_risk"   in tools_used else "skipped"
    sv          = state.get("safety_verdict") or {}
    verdict     = (sv.get("output") or sv).get("verdict", "None")
    path_type   = "escalated" if state.get("is_emergency") else "safety_evaluated"

    print(f"    memory: record #{idx}" if idx >= 0
          else "\n · memory: skipped (critique fail — would pollute recall)")
    print("\n  ─── FINALIZE " + "─" * 52)
    print(f"    decision path: {path_type} · d3={d3_status} · allergy={ally_status} · policy: {verdict}")
    return {}


def route_after_orchestrator(state: CrisisOrchestratorState) -> str:
    if state.get("is_emergency"):
        return "emergency_alert"
    if state.get("skip_critique"):
        return "finalize"
    return "critique"


# Graph compile

def route_after_critique(state: CrisisOrchestratorState) -> str:
    """Route back to orchestrator for remediation, or finalize."""
    if state.get("critique_feedback"):
        return "orchestrator"
    return "finalize"

### 11.2 · Compile the graph

In [ ]:
checkpointer = InMemorySaver()
wf = StateGraph(CrisisOrchestratorState)

wf.add_node("intake",          intake_node)
wf.add_node("orchestrator",    orchestrator_node)
wf.add_node("critique",        critique_node)
wf.add_node("emergency_alert", emergency_alert_node)
wf.add_node("finalize",        finalize_node)

wf.add_edge(START,         "intake")
wf.add_edge("intake",      "orchestrator")
wf.add_conditional_edges(
    "orchestrator",
    route_after_orchestrator,
    {"critique": "critique", "emergency_alert": "emergency_alert", "finalize": "finalize"},
)
wf.add_conditional_edges(
    "critique",
    route_after_critique,
    {"orchestrator": "orchestrator", "finalize": "finalize"},
)
wf.add_edge("emergency_alert", "finalize")
wf.add_edge("finalize",        END)

graph = wf.compile(checkpointer=checkpointer)
print("LangGraph compiled.")
print("Nodes:", list(graph.get_graph().nodes.keys()))

## 12 · Run Summary, Logging & NL Report

- **`run_trace(label, case_input, verbose=False, skip_critique=False)`** — parses
  input, invokes the graph, prints the summary, returns the final state.
- **`print_run_summary`** — two blocks: **SYSTEM AUDIT** (path · tool count · elapsed;
  per-tool timeline with a `_conf_bar` label `[HIGH]`≥0.90 / `[ OK ]`≥0.70 /
  `[LOW ]`≥0.50 / `[POOR]`<0.50; critique line) and **CLINICAL RECOMMENDATION**
  (urgency · verdict · summary, or the escalation payload on the emergency path).


### 12.1 · Formatting helpers

In [ ]:
import textwrap

W   = 64
BAR = "=" * W
SUB = "-" * W


def _status(conf: float) -> str:
    """Confidence tier label."""
    return ("HIGH" if conf >= 0.90 else
            "OK"   if conf >= 0.70 else
            "LOW"  if conf >= 0.50 else
            "POOR")


def _wrap(text: str, indent: int, first: str = "") -> None:
    """Print text wrapped to W, with an optional first-line label."""
    pad   = " " * indent
    lines = textwrap.wrap(str(text).replace("\n", " "), W - indent) or [""]
    print(f"{' ' * (indent - len(first) - 1)}{first} {lines[0]}" if first
          else f"{pad}{lines[0]}")
    for ln in lines[1:]:
        print(f"{pad}{ln}")

### 12.2 · Structured run summary

In [ ]:
def print_run_summary(state: dict, elapsed: float, verbose: bool = False) -> None:
    """Two clearly separated blocks: RUN SUMMARY (audit) and
    CLINICAL RECOMMENDATION (clinician-facing). No content is truncated."""
    trace    = state.get("audit_trace", [])
    is_emerg = state.get("is_emergency", False)
    cr       = state.get("critique_report") or {}
    cn       = state.get("clinical_note") or {}
    cn_out   = cn.get("output", cn)
    path     = "EMERGENCY BYPASS" if is_emerg else "CONTINUE (safety-evaluated)"

    # Block 1: Run summary / audit
    print()
    print(BAR)
    print(f"  RUN SUMMARY   ·   {path}")
    print(f"  {len(trace)} tool call(s)   ·   {elapsed:.1f}s")
    print(BAR)

    if trace:
        print("  TOOL TIMELINE")
        print(f"    {'#':<3} {'tool':<26} {'conf':<6} {'status':<8} notes")
        for i, hd in enumerate(trace, 1):
            skipped = "Skipped" in hd.get("reasoning_trace", "")
            conf    = hd.get("confidence", 0.0)
            cf, st  = ("  -", "SKIPPED") if skipped else (f"{conf:.2f}", _status(conf))
            notes   = []
            if not skipped:
                if conf < 0.70:
                    notes.append("low-confidence")
                if hd.get("caveats"):
                    notes.append(f"{len(hd['caveats'])} caveat(s)")
            print(f"    {i:<3} {hd['tool_name']:<26} {cf:<6} {st:<8} {', '.join(notes)}")
            if skipped:
                continue
            if verbose and hd.get("reasoning_trace"):
                for ln in textwrap.wrap(hd["reasoning_trace"], W - 9):
                    print(f"        {ln}")
            for cav in hd.get("caveats", []):
                _wrap(cav, 9, first="      caveat:")
        print()

    if cr and not cr.get("bypass"):
        passed = cr.get("passed", True)
        print(f"  CRITIQUE   ·   {'PASS' if passed else 'FAIL'}   ·   "
              f"completeness {cr.get('audit_completeness', 0.0):.0%}   ·   "
              f"{cr.get('recommendation', '?')}")
        for c in cr.get("checks", []):
            mark  = "PASS" if c.get("passed") else "FAIL"
            issue = c.get("issue") or ""
            print(f"    [{mark}] {c['name']}" + (f" — {issue}" if issue else ""))
        if cr.get("critique_summary"):
            for ln in textwrap.wrap(cr["critique_summary"], W - 6):
                print(f"      {ln}")
        print()

    # Block 2: Clinical recommendation
    print(SUB)
    print("  CLINICAL RECOMMENDATION")
    print(SUB)

    if is_emerg:
        ep = state.get("emergency_payload", {})
        print(f"  ESCALATION   {ep.get('escalation_level', 'ICU-immediate').upper()}")
        if ep.get("reason"):
            _wrap(ep["reason"], 9, first="  REASON:")
        if ep.get("immediate_actions"):
            print("  IMMEDIATE ACTIONS")
            for a in ep["immediate_actions"]:
                _wrap(a, 6, first="    -")
    else:
        tro = (state.get("triage_result") or {}).get("output",
              state.get("triage_result") or {})
        svo = (state.get("safety_verdict") or {}).get("output",
              state.get("safety_verdict") or {})
        # Structured display: split the verdict into class / disposition /
        # rationale instead of cramming everything onto one line.
        _urg = tro.get("urgency", "unknown").upper()
        _vraw = (svo.get("verdict") or "").strip()
        _rc = svo.get("recommendation_class") or ""
        _od = svo.get("order_disposition") or ""
        # Fallback: parse from combined `verdict` if structured fields are missing.
        if (not _rc or not _od) and " — " in _vraw:
            _parts = _vraw.split(" — ", 1)
            _rc = _rc or _parts[0].strip()
            _od = _od or _parts[1].split(";", 1)[0].split(".", 1)[0].strip()
        _CLASS_LABEL = {
            "Class_I":         "Class I  (Strong — should be performed)",
            "Class_IIa":       "Class IIa (Reasonable to perform)",
            "Class_IIb":       "Class IIb (May be considered)",
            "Class_III_NB":    "Class III: No Benefit",
            "Class_III_Harm":  "Class III: Harm",
        }
        _DISP_LABEL = {
            "approve_as_written":         "Approve as written",
            "approve_with_modification":  "Approve with modification",
            "hold_pending_clarification": "Hold pending clarification",
            "reject":                     "Reject",
        }
        _class_disp = _CLASS_LABEL.get(_rc, _rc or "—")
        _disp_disp  = _DISP_LABEL.get(_od, _od.replace("_", " ").capitalize() if _od else "—")
        _reason = (svo.get("verdict_reasoning") or "").strip()
        print(f"  URGENCY        {_urg}")
        print(f"  CLASS          {_class_disp}")
        print(f"  DISPOSITION    {_disp_disp}")
        if _reason:
            _wrap(_reason, 17, first="  RATIONALE     ")
        print()
        # Build a lookup of per-action safety annotations from the audit trace.
        # We cross-reference each approved_action against the d3 caution_actions
        # (drug-drug interactions) and allergy high/moderate/low-risk lists.
        _d3_o   = next((h.get("output", {}) or {} for h in trace
                        if h.get("tool_name") == "screen_drug_conflicts"), {})
        _ally_o = next((h.get("output", {}) or {} for h in trace
                        if h.get("tool_name") == "assess_allergy_risk"), {})
        _rag_o  = next((h.get("output", {}) or {} for h in trace
                        if h.get("tool_name") == "retrieve_clinical_data"), {})
        _d3_flagged   = {(c.get("action", "") or "").lower()
                         for c in _d3_o.get("caution_actions", [])}
        _ally_flagged = set()
        for k in ("high_risk_actions", "moderate_risk_actions"):
            for it in _ally_o.get(k, []):
                _ally_flagged.add((it.get("action", "") or "").lower())
        _drug_pool = {p.get("drug", "").lower(): p
                      for p in (_rag_o.get("drug_patterns") or [])}
        _has_meds  = bool((state.get("case_state") or {}).get("medications"))
        _has_allgs = bool((state.get("case_state") or {}).get("allergies"))

        def _safety_annot(action_text: str) -> str:
            tl = (action_text or "").lower()
            # DDI status
            if not _has_meds:
                ddi = "n/a (no current meds)"
            elif any(f and f in tl for f in _d3_flagged):
                ddi = "caution flagged"
            else:
                ddi = "cleared"
            # Allergy status
            if not _has_allgs:
                allg = "n/a (no allergies)"
            elif any(f and f in tl for f in _ally_flagged):
                allg = "caution flagged"
            else:
                allg = "cleared"
            # Evidence (RAG drug-pattern match)
            ev = ""
            for drug, pat in _drug_pool.items():
                if drug and drug in tl:
                    ev = f" · evidence: {pat["count"]}/5 similar cases"
                    break
            return f"DDI {ddi} · allergy {allg}{ev}"

        if cn_out.get("approved_actions"):
            print("  RECOMMENDED ACTIONS")
            for j, a in enumerate(cn_out["approved_actions"], 1):
                txt = a.get("action", str(a)) if isinstance(a, dict) else str(a)
                _wrap(txt, 7, first=f"    {j}.")
                print(f"       └─ {_safety_annot(txt)}")
            _n_excl = len((state.get("safety_verdict") or {}).get(
                "output", {}).get("excluded_actions", []) or [])
            if _n_excl:
                print(f"    ⚠  {_n_excl} action(s) excluded for safety "
                      f"(see audit trace)")
            print()
        for label, key in (("MONITOR", "required_monitoring"),
                           ("STOP IF", "stop_conditions"),
                           ("FOLLOW-UP", "follow_up")):
            for it in cn_out.get(key, []):
                _wrap(it, 6, first=f"  {label}:" if label else "    -")
            if cn_out.get(key):
                print()
        alert = cn_out.get("clinician_alert", "")
        if alert and alert.strip():
            print("  ** CLINICIAN ALERT **")
            for ln in textwrap.wrap(alert, W - 6):
                print(f"    {ln}")
            print()

    # Similar patients — every retrieved case, full excerpt (no truncation)
    rag_cases = next((hd.get("output", {}).get("cases", [])
                      for hd in trace if hd.get("tool_name") == "retrieve_clinical_data"),
                     [])
    # (Similar Patients block omitted — already shown live during the
    # retrieve_clinical_data call; printing again here was noise.)
    print(BAR)

### 12.3 · `run_trace` — per-case entry point

In [ ]:
def run_trace(label: str, case_input: Any, verbose: bool = False,
              skip_critique: bool = False) -> dict:
    """Run the full graph. Streams the live execution log, then prints the
    structured RUN SUMMARY + CLINICAL RECOMMENDATION. Returns the final state.

    skip_critique=True routes orchestrator -> finalize directly (bypasses the
    Critique Agent — useful for latency testing)."""
    print()
    print(BAR)
    print(f"  CRISIS CDS   ·   {label}")
    print(f"  started {datetime.now().strftime('%H:%M:%S')}")
    print(BAR)

    cs, missing = parse_intake(case_input)
    thread_id   = str(uuid.uuid4())
    config      = {"configurable": {"thread_id": thread_id}}

    meds  = [m.get("name", m) if isinstance(m, dict) else m for m in cs.get("medications", [])]
    allgs = [a.get("allergen", a) if isinstance(a, dict) else a for a in cs.get("allergies", [])]
    syms  = cs.get("symptoms", [])
    print(f"  INTAKE  symptoms={len(syms)} · vitals={len(cs.get('vitals', {}))} · "
          f"meds={len(meds)} · allergies={len(allgs)}"
          + (f"  [{len(missing)} missing]" if missing else ""))
    if verbose:
        if syms:  print(f"          symptoms : {syms}")
        if meds:  print(f"          meds     : {meds}")
        if allgs: print(f"          allergies: {allgs}")
        if missing: print(f"          missing  : {missing}")

    print()

    initial_state = {
        "case_state":         cs,
        "missing_data_flags": missing,
        "thread_id":          thread_id,
        "messages":           [],
        "audit_trace":        [],
        "skip_critique":      skip_critique,
    }
    t0 = time.time()
    try:
        final_state = graph.invoke(initial_state, config=config)
    finally:
        # Flush this case's checkpoint so InMemorySaver does not retain every
        # case's full state (messages + tool outputs) for the whole batch.
        try:
            checkpointer.delete_thread(thread_id)
        except Exception:
            pass
    print_run_summary(final_state, time.time() - t0, verbose=verbose)
    return final_state

print("Logging helpers defined.")

## End-to-End Demo

Pick a case from `eval_1000_real.csv` (by `ROW` or `NOTE_ID`) and run the whole pipeline on its narrative. `run_trace()` streams the live system log and prints the system's own structured **SYSTEM AUDIT** (tool timeline, per-tool reasoning with `verbose=True`, critique block) and **CLINICAL RECOMMENDATION**. The returned `state` is the structured result.


In [ ]:
ENABLE_DEMO = False   # flip True to run one full end-to-end case (~150s, real tokens)

import os as _os
if _os.environ.get("CDS_IMPORT_ONLY") == "1" or not ENABLE_DEMO:
    print(f"End-to-End Demo skipped (ENABLE_DEMO={ENABLE_DEMO}). Flip to True to run.")
else:
    # ════════════════════════════ END-TO-END DEMO ════════════════════════════
    # Choose a case: set ROW (0-based) OR NOTE_ID (takes priority if not None).
    ROW     = 0
    NOTE_ID = None        # e.g. "14293608-DS-6"

    _df = pd.read_csv(DATA_DIR / "eval_1000_real.csv", encoding="utf-8-sig")
    _r  = (_df[_df["note_id"].astype(str) == str(NOTE_ID)].iloc[0]
           if NOTE_ID is not None else _df.iloc[ROW])

    print(f"CASE {_r['note_id']}  |  ground truth: urgency={_r['urgency_tier']} "
          f"(real ESI {_r['esi_acuity']})  |  ED cc: {_r['ed_chief_complaint']}")

    state = run_trace(f"{_r['note_id']} - ESI{_r['esi_acuity']}/{_r['urgency_tier']}",
                      str(_r["patient_narrative"]), verbose=True)